# RT-DETR 2차 실험 (AI Hub 데이터 추가)

## 목적

- 1차 RT-DETR 베이스라인(`jaeone-rtdetr-baseline.ipynb`)과 같은 학습 설정으로, 학습 데이터만 2차 실험의 방법(v2, v3, v4)으로 바꿔 학습한다.
- 세 방법 중 어느 방법이 성능이 더 좋은지 비교한다. 데이터가 크게 늘어서 epoch는 줄이고, 방법마다 epoch를 2~3가지로 바꿔 돌린다.
- 진행은 팀 2차 실험 가이드(`EXPERIMENTS.md`)의 "로컬에서 학습" 절 4~8단계를 따른다.

## 데이터

| 방법 | 뜻 | 모델 클래스 | train 사진 | train 객체 |
|---|---|---:|---:|---:|
| v2 | 클래스별 200개를 목표로 고른 사진 | 118 | 7,129 | 26,734 |
| v3 | 대회 클래스가 든 사진만 | 96 | 8,974 | 33,872 |
| v4 | 사진 전부, 대회 클래스 밖 알약은 `others` 한 클래스 | 75 | 11,206 | 42,800 |

- 표는 `EXPERIMENTS.md`의 값이다. v1(사진 전부, 118 클래스)은 팀에서 이번 실험 후보에서 뺐다.
- val은 세 방법이 같은 1,000장(Kaggle 47장 + AI Hub 953장)이다. Kaggle 47장은 1차 val과 같다.

## 목차

| 섹션 | 내용 |
|---|---|
| 0. 준비 | 레포 최상위로 이동, 경로, W&B 팀 설정 |
| 1. 학습 설정 | 1차 설정을 직접 적은 공통값, 1차와 값이 다른 것과 그 이유, run 이름, 학습 함수 |
| 2. 학습 폴더 확인 | 방법별 학습 폴더의 클래스 수, 사진 수, 객체 수 |
| 3. 채점 준비 | 팀 채점 함수로 점수 3개(val, kaggle, test_only)를 내는 함수, 1차 모델 다시 채점 |
| 4. 시간 측정 | v2로 1 epoch를 학습해 epoch당 시간을 재고 본 학습의 epoch를 정한다, 채점 시험 |
| 5. 본 학습과 채점 | 5 epoch, 10 epoch 순서로 v2, v3, v4를 학습하고 run마다 채점한다 |
| 6. 결과 기록 | `runs.csv` |
| 7. 결과 비교 | 방법과 epoch별 점수 비교 |
| 8. 제출 | val로 고른 run만 Kaggle 제출 CSV를 만든다 |

## 실행 방법

- 레포를 최신 main으로 맞춘 뒤(`git pull origin main`, `uv sync`) 실험 브랜치에서 실행한다.
- 학습 폴더는 터미널에서 미리 만든다(2번).
- 커널을 새로 띄우면 0번을 가장 먼저 실행한다. 0번보다 먼저 Ultralytics를 쓰면 W&B 기록이 팀 프로젝트에 시작되지 않아서, 학습이 첫 epoch 전에 멈춘다.

## 0. 준비

- 노트북이 `notebooks/modeling/` 아래에 있어도 레포 최상위 폴더로 이동한다. 그래서 아래 경로는 모두 레포 최상위 기준의 상대경로다. `EXPERIMENTS.md` 5단계의 준비 셀과 같은 방식이다.
- W&B
    - 2차 실험은 가이드대로 Ultralytics에 내장된 W&B 기록을 쓴다. 1차 노트북처럼 `wandb.init`을 직접 부르지 않는다.
    - `WANDB_ENTITY`로 팀(`codeit-prj1-team2`)을 정하고, Ultralytics 설정의 `wandb`를 켠다. 이 설정은 이 컴퓨터의 Ultralytics 설정 파일에 저장되어 다른 노트북에도 남는다.
    - W&B 프로젝트 이름은 학습할 때 `project="pill-object-detection"`으로 주고, 실행 이름은 run 폴더 이름이 된다(1번).
    - `require_wandb`는 첫 epoch 직전에 불려서, W&B 기록이 팀 프로젝트에 시작되지 않았으면 학습을 멈춘다. 시작됐으면 태그 `exp2`와 방법(v2 등)을 붙인다.
- 사전학습 가중치는 1차와 같은 `weights/rtdetr-l.pt`를 쓴다.

In [ ]:
import json
import math
import os
import platform
import shutil
import sys
import time
from pathlib import Path

import pandas as pd
import torch
import ultralytics
import wandb
import yaml
from ultralytics import RTDETR, settings

from pill_detection.evaluate import evaluate_val, predictions_from_ultralytics  # 팀 공용 채점 함수 (3번)

# 레포 최상위로 이동한다. 아래 경로는 모두 레포 최상위 기준 (EXPERIMENTS.md 5단계 준비 셀과 같다)
os.chdir(next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file()))

EXP2_DIR = Path("data/processed/exp2")                    # 방법별 학습 폴더 (2번에서 확인)
RUNS_EXP2 = Path("runs/exp2")                             # 2차 run 폴더
RUNS_EXP1 = Path("runs")                                  # 1차 run 폴더 (rtdetr-l_e100_s*_fix)
WEIGHTS = Path("weights/rtdetr-l.pt")                     # COCO 사전학습 가중치 (1차와 같은 파일)
RECORD_DIR = Path("notebooks/modeling/jaeone-rtdetr-exp2")  # 결과 기록 runs.csv를 둘 곳 (6번)
assert WEIGHTS.exists(), f"사전학습 가중치가 없습니다: {WEIGHTS.as_posix()}"

# W&B 팀 프로젝트 codeit-prj1-team2/pill-object-detection에 기록한다
os.environ["WANDB_ENTITY"] = "codeit-prj1-team2"
settings.update({"wandb": True})


def require_wandb(trainer):
    """첫 epoch 직전에 불린다. W&B 기록이 팀 프로젝트에 시작되지 않았으면 학습을 멈추고, 시작됐으면 2차 실험 태그를 붙인다."""
    if wandb.run is None:
        raise RuntimeError("W&B 기록이 시작되지 않았습니다. 커널을 다시 시작하고, Ultralytics를 쓰기 전에 0번 셀부터 실행하세요.")
    if wandb.run.entity != "codeit-prj1-team2":
        raise RuntimeError(f"W&B 기록이 팀이 아닌 {wandb.run.entity}에 시작됐습니다. 커널을 다시 시작하고 0번 셀부터 실행하세요.")
    wandb.run.tags = sorted({*wandb.run.tags, "exp2", Path(trainer.args.data).parent.name})  # data.yaml이 든 폴더 이름 = 방법


DEVICE = 0 if torch.cuda.is_available() else "cpu"  # 0번 GPU. GPU가 없으면 CPU

print("작업 폴더   : 레포 최상위" if Path("pyproject.toml").is_file() else "작업 폴더를 찾지 못했습니다")
print("OS         :", platform.system())
print("Python     :", sys.version.split()[0])
print("PyTorch    :", torch.__version__, "/ CUDA:", torch.version.cuda)
print("Ultralytics:", ultralytics.__version__, "" if ultralytics.__version__ == "8.4.183" else "(1차와 버전이 다릅니다: 1차는 8.4.183)")
print("W&B        :", wandb.__version__)
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print("GPU        :", gpu.name, f"/ 메모리 {gpu.total_memory / 1024**3:.1f} GiB")
else:
    print("GPU        : 없음 (CPU로 학습하면 매우 느립니다)")

## 1. 학습 설정

### 1-1. 공통값

- 팀 공통값은 따로 없고, 1차 실험의 설정을 그대로 쓴다. 다만 `EXPERIMENTS.md`에 따라 `optimizer`와 `lr0`를 직접 적는다.
    - `optimizer=auto`로 두면 Ultralytics가 클래스 수와 학습량으로 optimizer와 학습률을 고른다.
    - 2차는 방법마다 클래스 수가 달라서(118, 96, 75) 학습률이 방법마다 달라진다.
    - 학습량이 많으면(반복 1만 번 초과) AdamW 대신 MuSGD(lr 0.01)를 고른다.
    - 그러면 방법끼리 비교할 수 없고, 1차와도 달라진다.
- 1차에서 `auto`가 실제로 고른 값을 그대로 적는다. 1차 실험 조건 문서(`jaeone-rtdetr-exp1-conditions.ipynb`)에서 확인한 값이다.
    - `auto`는 lr0뿐 아니라 momentum(AdamW의 beta1)과 warmup_bias_lr도 바꿨다. lr0만 적으면 momentum은 기본값 0.937, warmup_bias_lr는 0.1이 되어 1차와 달라진다. 그래서 네 값을 함께 적는다.

| 인자 | 2차 값 | 1차 | 비고 |
|---|---|---|---|
| optimizer | AdamW | auto → AdamW | 직접 적는다 |
| lr0 | 0.000167 | auto가 고른 값 | 0.002 × 5 ÷ (4 + 1차 클래스 56) |
| momentum | 0.9 | auto가 정한 값 | AdamW의 beta1 |
| warmup_bias_lr | 0.0 | auto가 정한 값 | |
| imgsz | 640 | 같음 | 사진을 640×640으로 늘려 맞춘다 |
| batch | 16 | 같음 | |
| seed | 0 | 0, 1, 2 | 팀이 seed 0으로 통일했다 |
| patience | 30 | 같음 | epoch가 30 이하라 일찍 멈추는 일은 없다 |
| workers | 0 | 같음 | Windows 주피터에서 사진 읽기 오류를 피한다 |
| deterministic | False | 같음 | RT-DETR의 grid_sample이 지원하지 않는다 |
| cache | False | 같음(기본값) | 가이드대로 적는다 |
| epochs | 5, 10 등 | 100 | 데이터가 38~61배라 줄인다. 4번에서 시간을 재고 정한다 |
| warmup_epochs | 0.03 × epochs | 3 | 1-2 |
| close_mosaic | max(1, round(0.1 × epochs)) | 10 | 1-2 |
| 나머지 | Ultralytics 기본값 | 같음 | |

### 1-2. 1차와 값이 다른 것: warmup과 모자이크

- 두 설정은 epoch 수로 정해져 있다. epoch를 100에서 5나 10으로 줄이면서 1차 값(3, 10)을 그대로 두면, 실제로는 1차와 전혀 다른 학습이 된다. 그래서 값 대신 1차의 비율을 지킨다.
- warmup: 학습 초반에 학습률을 0에서 lr0까지 천천히 올리는 구간이다. 새로 붙인 층(클래스 점수를 내는 층)이 무작위 값이라, 처음부터 큰 학습률로 고치면 사전학습한 가중치까지 망가뜨릴 수 있어서 처음 몇십~몇백 번의 업데이트를 살살 한다.
    - Ultralytics 규칙: warmup 길이 = min(warmup_epochs, epochs − 1) × epoch당 batch 수
    - 1차: 3 epoch × 12 batch = 36 batch로, 전체 100 epoch의 3%였다.
    - 2차에 3을 그대로 쓰면 v2(epoch당 446 batch)는 1,338 batch가 warmup이다. 5 epoch면 학습의 60%, 10 epoch면 30%가 학습률을 올리는 구간이 된다.
    - 그래서 1차와 같은 비율(3%)로 `warmup_epochs = 0.03 × epochs`를 쓴다. 5 epoch면 0.15 epoch(v2 약 67 batch), 10 epoch면 0.3 epoch(약 134 batch)로, batch 수로도 1차(36 batch)에 가깝다.
- 모자이크: 사진 4장을 이어 붙여 한 장으로 만드는 증강이다. Ultralytics는 마지막 `close_mosaic` epoch 동안 모자이크를 끄고, 원래 모양의 사진으로 마무리한다.
    - Ultralytics 규칙: epoch ≥ epochs − close_mosaic이 되는 순간부터 끈다.
    - 1차: 100 epoch 중 90 epoch는 모자이크를 쓰고, 마지막 10 epoch(10%)는 껐다.
    - 2차에 10을 그대로 쓰면 5나 10 epoch에서는 "epochs − 10"이 0 이하라, 처음부터 끝까지 모자이크를 쓰지 않는다.
    - 그래서 1차와 같은 비율(10%)로 `close_mosaic = max(1, round(0.1 × epochs))`를 쓴다. 정수여야 해서 5 epoch와 10 epoch 모두 마지막 1 epoch만 끈다(5 epoch는 20%, 10 epoch는 10%).
- 이렇게 하면 5 epoch와 10 epoch run을 비교할 때, 달라지는 것이 epoch 수뿐이다. 값을 그대로 두면 warmup 비율(60%와 30%)까지 함께 달라진다.
- 이 비율이 이번 데이터에서 가장 좋은 값인지는 확인하지 않았다. 1차와 같은 모양으로 학습하기 위한 선택이다.

| | 1차 (185장, 100 epoch) | 2차 v2, 5 epoch | 2차 v2, 10 epoch |
|---|---|---|---|
| warmup | 3 epoch = 36 batch (3%) | 0.15 epoch ≈ 67 batch (3%) | 0.3 epoch ≈ 134 batch (3%) |
| 모자이크 | 90 epoch 사용, 마지막 10 epoch 끔 | 4 epoch 사용, 마지막 1 epoch 끔 | 9 epoch 사용, 마지막 1 epoch 끔 |

### 1-3. run 이름과 학습 함수

- run 이름은 가이드의 `<방법>_<모델>` 뒤에 epoch를 붙인다(예: `v2_rtdetr-l_e5`). 같은 방법을 epoch만 바꿔 여러 번 돌리기 때문이다. run 폴더는 `runs/exp2/<run 이름>`이고, 폴더 이름이 W&B 실행 이름이 된다.
- 시간 측정 run(4번)은 가이드의 스모크 규칙대로 이름 앞에 `smoke_`를 붙이고 W&B 태그 `smoke`를 단다. 본 학습과 비교하지 않는다.
    - 시간 측정 run은 모자이크를 끄지 않는다(`close_mosaic=0`). 모자이크는 사진 4장을 읽어서 한 장을 만들기 때문에 사진 읽는 시간이 늘어난다. 본 학습은 대부분의 epoch에서 모자이크를 쓰므로, 같은 조건에서 재야 시간이 맞게 나온다.
- `train_one`은 같은 이름의 run 폴더가 이미 있으면 멈춘다. 이어서 학습할 때는 5번의 이어서 학습 셀을 쓴다.
- `train_one`의 `callbacks`에는 Ultralytics 콜백(이벤트 이름, 함수)을 더 줄 수 있다. 4번에서 시간을 잴 때 쓴다.
- 학습 중에 Ultralytics가 출력하는 mAP50-95로는 방법끼리 비교하지 않는다. 대회 클래스 밖 클래스까지 평균하고, IoU 기준도 대회(0.75~0.95)와 다르다. 비교는 3-1의 채점 함수로 낸 점수 3개로 한다(`EXPERIMENTS.md` 5단계).

In [ ]:
MODEL_NAME = "rtdetr-l"
COMMON = dict(  # 1차 설정 (1-1)
    optimizer="AdamW", lr0=0.000167, momentum=0.9, warmup_bias_lr=0.0,
    imgsz=640, batch=16, seed=0, patience=30,
    workers=0,            # Windows 주피터에서 사진 읽기 오류를 피하려고 0
    deterministic=False,  # RT-DETR의 grid_sample이 deterministic을 지원하지 않는다
    cache=False,
)


def run_name(variant, epochs, smoke=False):
    """run 폴더 이름 = W&B 실행 이름. 예: v2_rtdetr-l_e5, 시간 측정은 smoke_v2_rtdetr-l_e1"""
    name = f"{variant}_{MODEL_NAME}_e{epochs}"
    return f"smoke_{name}" if smoke else name


def epoch_args(epochs, smoke=False):
    """epoch 수로 정해지는 값 (1-2). 1차의 비율을 지킨다."""
    return dict(epochs=epochs,
                warmup_epochs=round(0.03 * epochs, 3),                        # 1차: 100 epoch 중 3 epoch (3%)
                close_mosaic=0 if smoke else max(1, round(0.1 * epochs)))     # 1차: 100 epoch 중 마지막 10 epoch (10%). 시간 측정은 끄지 않는다


def train_args(variant, epochs, smoke=False):
    """model.train()에 넘길 인자. 여기 없는 인자는 Ultralytics 기본값을 쓴다."""
    return dict(data=str(EXP2_DIR / variant / "data.yaml"),
                project="pill-object-detection",                             # W&B 프로젝트 이름
                save_dir=str(RUNS_EXP2 / run_name(variant, epochs, smoke)),  # 결과 폴더. 폴더 이름이 W&B 실행 이름이 된다
                device=DEVICE, **COMMON, **epoch_args(epochs, smoke))


def mark_smoke(trainer):
    """require_wandb 다음에 불린다. 시간 측정 run임을 W&B 태그로 남긴다."""
    wandb.run.tags = sorted({*wandb.run.tags, "smoke"})


def train_one(variant, epochs, smoke=False, callbacks=()):
    """방법 하나, epoch 수 하나로 학습한다. 같은 이름의 run 폴더가 있으면 멈춘다. callbacks: (이벤트 이름, 함수) 목록"""
    args = train_args(variant, epochs, smoke)
    run_dir = Path(args["save_dir"])
    assert not run_dir.exists(), f"{run_dir.as_posix()}이 이미 있습니다. 이어서 학습이면 5번의 이어서 학습 셀을, 새 실험이면 run 이름을 바꾸세요."
    model = RTDETR(str(WEIGHTS))
    model.add_callback("on_train_start", require_wandb)
    if smoke:
        model.add_callback("on_train_start", mark_smoke)
    for event, fn in callbacks:
        model.add_callback(event, fn)
    model.train(**args)
    print("run 폴더:", run_dir.as_posix())
    return run_dir


# 예시: v2를 5 epoch, 10 epoch로 돌릴 때와 시간 측정 run의 인자
for v, e, s in [("v2", 5, False), ("v2", 10, False), ("v2", 1, True)]:
    a = train_args(v, e, s)
    print(f"{run_name(v, e, s):22s} epochs={a['epochs']:<3} warmup_epochs={a['warmup_epochs']:<5} close_mosaic={a['close_mosaic']}")
print()
for k, v in train_args("v2", 5).items():
    print(f"{k:15}: {v}")

## 2. 학습 폴더 확인

- 학습 폴더는 가이드 4단계대로 터미널(레포 최상위)에서 방법마다 한 번 만든다. 사진은 복사하지 않고 원본에 하드링크를 건다.
    ```
    uv run scripts/build_dataset.py --variant v2 --out data/processed/exp2/v2
    uv run scripts/build_dataset.py --variant v3 --out data/processed/exp2/v3
    uv run scripts/build_dataset.py --variant v4 --out data/processed/exp2/v4
    ```
- 이 셀은 만든 폴더의 `dataset_info.json`과 `data.yaml`을 읽어, 클래스 수, train 사진 수, train 객체 수가 `EXPERIMENTS.md`의 표와 같은지 확인한다. 같으면 팀과 같은 데이터로 학습하는 것이다.
- val은 세 방법이 같은 1,000장이어야 한다. val 객체 수는 방법마다 다를 수 있다. 그 방법의 모델 클래스가 아닌 객체(예: v3에 없는 클래스)는 빼고 셌기 때문이다(`datasets.py` 설명).
- `git commit`은 학습 폴더를 만들 때의 레포 커밋이다. `EXPERIMENTS.md` 7단계의 결과 기록에 들어간다.

In [ ]:
EXPECTED = {"v2": (118, 7129, 26734), "v3": (96, 8974, 33872), "v4": (75, 11206, 42800)}  # EXPERIMENTS.md 표: 클래스, train 사진, train 객체

ds_rows = []
for v, (nc, n_img, n_obj) in EXPECTED.items():
    ds_dir = EXP2_DIR / v
    assert (ds_dir / "data.yaml").is_file(), (f"{ds_dir.as_posix()}에 학습 폴더가 없습니다. 터미널(레포 최상위)에서 "
                                              f"uv run scripts/build_dataset.py --variant {v} --out {ds_dir.as_posix()} 를 먼저 실행하세요.")
    info = json.loads((ds_dir / "dataset_info.json").read_text(encoding="utf-8"))
    names = yaml.safe_load((ds_dir / "data.yaml").read_text(encoding="utf-8"))["names"]
    ds_rows.append({"방법": v, "클래스": info["nc"], "train 사진": info["n_images"]["train"], "train 객체": info["n_objects"]["train"],
                    "val 사진": info["n_images"]["val"], "val 객체": info["n_objects"]["val"],
                    "클래스 이름 예": f"{names[0]}, ..., {names[len(names) - 1]}", "git commit": (info.get("git_commit") or "")[:7]})
    assert (info["nc"], info["n_images"]["train"], info["n_objects"]["train"]) == (nc, n_img, n_obj), f"{v}가 EXPERIMENTS.md 표와 다릅니다."
    assert info["n_images"]["val"] == 1000, f"{v}의 val이 1,000장이 아닙니다."
    assert len(names) == info["nc"], f"{v}의 data.yaml 클래스 수가 다릅니다."

display(pd.DataFrame(ds_rows).set_index("방법"))

## 3. 채점 준비

### 3-1. 채점 함수

- 2차 실험의 점수는 팀 공용 채점 함수(`pill_detection.evaluate.evaluate_val`)로 낸다. `EXPERIMENTS.md` 6단계와 같은 방법이다.
    - 지표는 대회 지표와 같은 mAP@[0.75:0.95]다. pycocotools로 계산한다.
    - 점수는 3개이고, 채점하는 범위만 다르다.

| 점수 | 범위 | 쓰는 곳 |
|---|---|---|
| `val` | val 1,000장 전체 | 방법과 epoch 비교의 기준 |
| `kaggle` | val 중 Kaggle 사진 47장(1차 val과 같은 사진) | 1차와 비교 |
| `test_only` | test에만 나오는 18개 클래스 | AI Hub로만 배운 클래스를 잘 찾는지 |

- 채점하지 않는 예측도 있다. 대회 클래스(74개) 밖의 약 번호로 낸 예측(`others` 포함)과, 대회 클래스 밖 알약에 IoU 0.5 이상 겹친 예측은 버린다. 자세한 규칙은 `evaluate.py` 맨 위 설명과 `data/datasets/README.md`에 있다.
- 예측 조건: best.pt로 val 1,000장을 예측한다. 입력 크기는 640, 점수 하한은 0.001이다(가이드 6단계. Ultralytics val의 기본값과 같다).
- 함수 두 개를 만든다.
    - `score_weights`: 가중치 하나로 예측하고 점수를 낸다. 아무것도 저장하지 않는다. 3-2에서 1차 모델을 잴 때 쓴다.
    - `score_run`: 2차 run 하나를 채점한다. 가이드 6단계의 일을 한 번에 한다.
        1. best.pt로 점수 3개를 낸다.
        2. run 폴더에 `scores.json`(점수 3개, 클래스별 AP)을 저장한다.
        3. 학습 폴더의 `dataset_info.json`을 run 폴더로 복사한다.
        4. run 폴더의 `wandb/` 기록에서 학습 때의 W&B 실행을 찾아, Summary에 `exp2/val`, `exp2/kaggle`, `exp2/test_only`를 붙인다. 팀은 W&B 실행 표에서 이 열로 방법과 모델을 비교한다.
- val은 세 방법이 같은 1,000장이라, 1차 모델처럼 어느 방법으로도 학습하지 않은 모델은 v2 폴더의 val로 잰다.

In [ ]:
SCORE_CONF = 0.001                         # 채점용 예측의 점수 하한 (EXPERIMENTS.md 6단계)
SCORE_KEYS = ["val", "kaggle", "test_only"]


def score_weights(weights, variant="v2"):
    """가중치 하나로 2차 val 1,000장을 예측하고 팀 채점 함수로 점수를 낸다. 저장은 하지 않는다."""
    ds_dir = EXP2_DIR / variant
    model = RTDETR(str(weights))
    results = model.predict(source=str(ds_dir / "images" / "val"), imgsz=COMMON["imgsz"], conf=SCORE_CONF, device=DEVICE,
                            stream=True, verbose=False)
    return evaluate_val(predictions_from_ultralytics(results), ds_dir / "eval" / "val_gt.json")


def find_wandb_run_id(run_dir):
    """run 폴더의 wandb/ 기록에서 학습 때의 W&B 실행 id를 찾는다. 여러 개면 가장 최근 것 (EXPERIMENTS.md 6단계와 같은 방법)"""
    found = max((Path(run_dir) / "wandb").glob("*run-*/run-*.wandb"), key=lambda f: f.stat().st_mtime, default=None)
    if found is None:
        raise FileNotFoundError(f"W&B 기록이 없습니다: {Path(run_dir).as_posix()}/wandb/\n"
                                "[해결] 0번 셀부터 실행한 커널에서 train_one으로 학습한 run인지 확인하세요.")
    return found.stem[4:]


def score_run(run_dir, variant):
    """2차 run 하나를 채점한다. 점수 3개 → scores.json 저장 → dataset_info.json 복사 → 학습 때의 W&B 실행 Summary에 점수를 붙인다."""
    run_dir = Path(run_dir)
    scores = score_weights(run_dir / "weights" / "best.pt", variant)
    (run_dir / "scores.json").write_text(json.dumps(scores, indent=2), encoding="utf-8")
    shutil.copy(EXP2_DIR / variant / "dataset_info.json", run_dir)
    info = json.loads((EXP2_DIR / variant / "dataset_info.json").read_text(encoding="utf-8"))
    with wandb.init(entity="codeit-prj1-team2", project="pill-object-detection", id=find_wandb_run_id(run_dir),
                    resume="allow", dir=str(run_dir)) as run:
        run.summary.update({f"exp2/{k}": scores[k] for k in SCORE_KEYS})
        run.config.update({"dataset_info": info})
    print(f"{run_dir.name}: " + "  ".join(f"{k} {scores[k]:.4f}" for k in SCORE_KEYS))
    return scores

### 3-2. 1차 모델 다시 채점

- 왜 하는가: 1차 노트북의 val 점수(mAP75-95, 4-7과 6-2)는 Ultralytics의 계산 방식으로 냈고, 2차 점수는 팀 채점 함수(pycocotools)로 낸다. 두 방식은 예측과 정답의 짝을 짓는 방법과 AP를 계산하는 세부 방식이 달라서, 같은 모델이라도 숫자가 조금 다를 수 있다. 1차 숫자를 그대로 2차와 비교하면, 데이터를 바꾼 효과와 계산 방식이 바뀐 효과가 섞인다.
- 그래서 1차 best.pt(seed 0, 1, 2)를 3-1의 함수로 다시 잰다. `EXPERIMENTS.md`의 "1차 모델 다시 채점하기"와 같다.
- 1차와 비교할 점수는 `kaggle`이다. 1차 val과 같은 Kaggle 사진 47장의 점수이기 때문이다.
    - 1차 모델은 1차 데이터의 56종만 배웠다. 그래서 `val`(AI Hub 사진이 든 1,000장)과 `test_only`(test에만 있는 18종)는 낮게 나오고, 2차와 비교하지 않는다.
    - 2차는 seed 0만 돌리므로 1차 seed 0의 `kaggle`이 기준점이다. seed 1, 2는 1차의 seed 차이가 이 채점 방식에서도 비슷한지 보려고 함께 잰다.
- 표에는 1차 노트북에서 Ultralytics로 잰 val mAP75-95(4-7)도 함께 적는다. `kaggle`과 같은 사진, 같은 IoU 범위라서, 두 계산 방식의 차이를 볼 수 있다.
- 1차 모델의 클래스 이름이 약 번호 숫자("1900" 등)여야 채점할 수 있다(가이드). 셀이 먼저 확인한다.
- 결과는 `runs/exp2/exp1_rescore.json`에 저장한다. 7번에서 2차 점수와 비교할 때 읽는다.
- 실행: 0~3-1을 실행한 뒤 실행한다. 학습 없이 예측만 해서 seed마다 몇 분이다.

In [ ]:
EXP1_ULTRALYTICS_MAP75_95 = {0: 0.9816, 1: 0.9803, 2: 0.9731}  # 1차 노트북 4-7, 6-2: Ultralytics로 잰 val 47장 mAP75-95

rescore_rows = []
for seed in [0, 1, 2]:
    weights = RUNS_EXP1 / f"rtdetr-l_e100_s{seed}_fix" / "weights" / "best.pt"
    assert weights.exists(), f"1차 best.pt가 없습니다: {weights.as_posix()}"
    first_name = RTDETR(str(weights)).names[0]
    assert str(first_name).isdigit(), f"1차 모델의 클래스 이름이 약 번호가 아닙니다: {first_name}"
    scores = score_weights(weights)
    rescore_rows.append({"seed": seed, **{k: scores[k] for k in SCORE_KEYS},
                         "1차 노트북 val mAP75-95 (Ultralytics)": EXP1_ULTRALYTICS_MAP75_95[seed]})
    print(f"seed {seed}: 클래스 이름 예 {first_name} / " + "  ".join(f"{k} {scores[k]:.4f}" for k in SCORE_KEYS))

rescore = pd.DataFrame(rescore_rows).set_index("seed").round(4)
RUNS_EXP2.mkdir(parents=True, exist_ok=True)
(RUNS_EXP2 / "exp1_rescore.json").write_text(rescore.reset_index().to_json(orient="records", force_ascii=False, indent=2),
                                             encoding="utf-8")
print("1차 모델을 2차 채점 함수로 잰 점수 (1차와 비교하는 점수는 kaggle)")
display(rescore)

## 4. 시간 측정

### 4-1. v2로 1 epoch 학습

- 왜 하는가: 2차 데이터는 1차보다 train 사진이 38~61배 많아서, 1차처럼 100 epoch를 돌릴 수 없다. v2로 1 epoch를 실제로 학습해서 걸리는 시간을 재고, 그 시간으로 본 학습의 epoch 수를 정한다.
- 돌리는 것: `smoke_v2_rtdetr-l_e1`
    - v2 train 전체(7,129장)로 1 epoch를 학습한다. 설정은 1번의 공통값과 같다.
    - 1-3대로 모자이크는 끄지 않는다(`close_mosaic=0`). 본 학습은 대부분의 epoch에서 모자이크를 쓰기 때문이다.
    - epoch가 1이라 warmup은 없다. Ultralytics는 warmup을 최대 "epochs − 1"까지만 한다.
    - W&B에는 `smoke` 태그가 붙은 실행으로 남는다. 본 학습과 비교하지 않는다.
- 시간을 네 구간으로 나눠 잰다. Ultralytics가 학습 중에 부르는 콜백에서 그때의 시각을 기록한다.

| 구간 | 무엇이 들어가나 | 시작과 끝 (콜백) |
|---|---|---|
| 준비 | 모델과 데이터 준비, 라벨 캐시 만들기(방법마다 첫 학습에서만), W&B 시작 | `on_pretrain_routine_start` → `on_train_start` |
| 학습 | 1 epoch의 batch 446개 | `on_train_epoch_start` → `on_train_epoch_end` |
| val | epoch 끝의 val 1,000장 채점, 결과와 가중치 저장 | `on_train_epoch_end` → `on_fit_epoch_end` |
| 마무리 | best.pt로 val을 한 번 더 채점(Ultralytics가 학습 끝에 자동으로 한다), 그래프 저장 | `on_fit_epoch_end` → `on_train_end` |

- 셀 전체 시간도 잰다. 위 네 구간 밖의 시간(가중치 불러오기, 학습 뒤 W&B에 best.pt 올리기 등)은 "그 밖"으로 따로 적는다.
- 학습과 val을 나눠 재는 이유: v3, v4는 train 사진 수만 다르고 val 1,000장은 같다. 그래서 학습 시간만 사진 수에 비례해 늘리고, val 시간은 그대로 두어 v3, v4의 시간을 짐작할 수 있다.
- GPU 메모리: 1차에서 batch 16이 전용 GPU 메모리(12GB)를 넘어 시스템 메모리를 같이 쓰면서 크게 느려진 적이 있다(1차 4번 결과). 학습 구간에서 PyTorch가 잡은 메모리의 최댓값을 함께 기록한다.
- 결과는 run 폴더의 `timing.json`에 저장한다.
- 실행: 0번, 1번을 실행한 뒤 실행한다. 1차 속도로 단순 계산하면 1 epoch에 30분~1시간 정도 걸린다(추정).

In [ ]:
timing = {}  # 콜백 이름 → 불린 시각 목록 (perf_counter, 초)


def mark(key):
    """그 시점의 시각을 timing[key]에 기록하는 콜백을 만든다. 학습 구간의 GPU 메모리 최댓값도 기록한다."""
    def callback(trainer):
        timing.setdefault(key, []).append(time.perf_counter())
        if torch.cuda.is_available() and key == "epoch_start":
            torch.cuda.reset_peak_memory_stats()
        if torch.cuda.is_available() and key == "train_end":
            timing.setdefault("gpu_peak_gib", []).append(torch.cuda.max_memory_reserved() / 1024**3)
    return callback


TIMING_EVENTS = [("on_pretrain_routine_start", "setup_start"), ("on_train_start", "train_start"),
                 ("on_train_epoch_start", "epoch_start"), ("on_train_epoch_end", "train_end"),
                 ("on_fit_epoch_end", "fit_end"), ("on_train_end", "end")]

cell_start = time.perf_counter()
smoke_dir = train_one("v2", 1, smoke=True, callbacks=[(event, mark(key)) for event, key in TIMING_EVENTS])
cell_s = time.perf_counter() - cell_start

n_train = json.loads((EXP2_DIR / "v2" / "dataset_info.json").read_text(encoding="utf-8"))["n_images"]["train"]
measured = {
    "variant": "v2", "n_train": n_train, "batches": math.ceil(n_train / COMMON["batch"]),
    "setup_s": timing["train_start"][0] - timing["setup_start"][0],
    "train_s": timing["train_end"][0] - timing["epoch_start"][0],
    "val_s": timing["fit_end"][0] - timing["train_end"][0],      # fit_end는 두 번 불린다: epoch 끝, 마무리 채점 끝
    "final_s": timing["end"][0] - timing["fit_end"][0],
}
measured["other_s"] = cell_s - (timing["end"][0] - timing["setup_start"][0])
measured["cell_s"] = cell_s
measured["gpu_peak_gib"] = max(timing.get("gpu_peak_gib", [0.0]))
measured["gpu_total_gib"] = torch.cuda.get_device_properties(0).total_memory / 1024**3 if torch.cuda.is_available() else 0.0
(smoke_dir / "timing.json").write_text(json.dumps(measured, indent=2), encoding="utf-8")

parts = [("준비", "setup_s"), ("학습", "train_s"), ("val", "val_s"), ("마무리", "final_s"), ("그 밖", "other_s"), ("셀 전체", "cell_s")]
display(pd.DataFrame({"분": [round(measured[k] / 60, 1) for _, k in parts]}, index=[name for name, _ in parts]))
print(f"학습: batch {measured['batches']}개, batch 하나에 {measured['train_s'] / measured['batches']:.2f}초")
print(f"GPU 메모리 최댓값 {measured['gpu_peak_gib']:.1f} GiB / 전용 메모리 {measured['gpu_total_gib']:.1f} GiB")
print("저장:", (smoke_dir / "timing.json").as_posix())

### 4-2. 채점 시험

- 왜 하는가: 5번은 셀 하나가 "학습 → 채점 → 다음 학습"을 밤새 이어서 한다. 채점(`score_run`)에서 오류가 나면 첫 run이 끝난 직후 셀이 멈추고, 나머지 run은 시작되지 않는다. 그래서 4-1에서 만든 `smoke_v2_rtdetr-l_e1`로 `score_run`을 끝까지 한 번 돌려 본다.
- 확인하는 것
    - 점수 3개가 숫자로 나오는지. 1 epoch짜리라 점수 자체는 의미가 없다.
    - run 폴더에 `scores.json`, `dataset_info.json`이 생기는지
    - W&B의 smoke 실행 Summary에 `exp2/val`, `exp2/kaggle`, `exp2/test_only`가 붙는지. W&B 서버에서 다시 읽어 확인한다.
- 이 W&B 실행에는 `smoke` 태그가 있어서 팀 비교표에 섞이지 않는다. 확인이 끝나면 W&B 실행과 run 폴더를 지워도 된다(가이드의 스모크 규칙).
- 실행: 0~3-1을 실행한 뒤 실행한다. 몇 분 걸린다.

In [ ]:
smoke_dir = RUNS_EXP2 / run_name("v2", 1, smoke=True)
smoke_scores = score_run(smoke_dir, "v2")

for name in ["scores.json", "dataset_info.json"]:
    print(f"{name:18s}", "있음" if (smoke_dir / name).exists() else "없음")
api_run = wandb.Api().run(f"codeit-prj1-team2/pill-object-detection/{find_wandb_run_id(smoke_dir)}")
print("W&B 실행 :", api_run.name, "/ 태그:", api_run.tags)
print("W&B Summary:", {f"exp2/{k}": api_run.summary.get(f"exp2/{k}") for k in SCORE_KEYS})

## 5. 본 학습과 채점

### 5-1. 실행 순서와 지금 상태

- 방법마다 5 epoch와 10 epoch를 돌린다. 5 epoch의 v2, v3, v4를 먼저 하고, 그다음 10 epoch의 v2, v3, v4를 한다. 짧은 run을 먼저 끝내서 세 방법의 비교 결과를 먼저 보기 위해서다.

| 순서 | run |
|---|---|
| 1 | `v2_rtdetr-l_e5` |
| 2 | `v3_rtdetr-l_e5` |
| 3 | `v4_rtdetr-l_e5` |
| 4 | `v2_rtdetr-l_e10` |
| 5 | `v3_rtdetr-l_e10` |
| 6 | `v4_rtdetr-l_e10` |

- epoch 수(5, 10)는 4-1에서 잰 시간으로 정했다. v2 1 epoch의 학습 시간(14.8분)을 방법별 train 사진 수에 비례해 늘리면 6개 run이 모두 약 14.6시간이 걸릴 것이라 예측했다.

- 5-2는 run마다 지금 상태를 보고 할 일을 정한다. 그래서 중지나 재부팅으로 멈춰도, 5-2를 다시 실행하면 멈춘 곳부터 이어서 간다.

| 상태 | 판단 기준 | 5-2가 하는 일 |
|---|---|---|
| 없음 | run 폴더가 없다 | 새로 학습한 뒤 채점 |
| 중간에 멈춤 | `last.pt`에 끝난 epoch 번호와 optimizer 상태가 있다 | 이어서 학습한 뒤 채점 |
| 학습 끝남 | `last.pt`의 epoch가 −1이다(학습이 끝나면 Ultralytics가 optimizer를 지우고 −1로 바꾼다), 또는 마지막 epoch까지 저장됐다 | 채점만 |
| 채점까지 끝남 | `scores.json`이 있다 | 건너뜀 |
| 확인 필요 | run 폴더는 있는데 `last.pt`가 없다(첫 epoch를 끝내기 전에 멈춤) | 그 run을 건너뛰고 알린다. 폴더를 지우고 다시 실행하면 새로 학습한다 |

- 끝난 run에 "이어서 학습"을 시키면 Ultralytics는 경고만 하고 새 학습을 시작한다. 그래서 `last.pt`를 열어 끝난 run과 멈춘 run을 먼저 구분한다.
- 이 셀은 상태만 보여 주고 학습하지 않는다. 5-2를 실행하기 전과 다시 시작하기 전에 확인용으로 쓴다.

In [ ]:
PLAN = [("v2", 5), ("v3", 5), ("v4", 5), ("v2", 10), ("v3", 10), ("v4", 10)]  # 5 epoch를 먼저 한다
STATE_TEXT = {"new": "없음", "partial": "중간에 멈춤", "trained": "학습 끝남 (채점 전)", "scored": "채점까지 끝남",
              "broken": "확인 필요 (last.pt 없음)"}


def run_state(variant, epochs):
    """run 하나의 상태와 끝난 epoch 수를 돌려준다. (상태 표는 5-1 설명)"""
    run_dir = RUNS_EXP2 / run_name(variant, epochs)
    if (run_dir / "scores.json").exists():
        return "scored", epochs
    if not run_dir.exists():
        return "new", 0
    last = run_dir / "weights" / "last.pt"
    if not last.exists():
        return "broken", 0
    ckpt = torch.load(last, map_location="cpu", weights_only=False)  # Ultralytics 체크포인트 (직접 만든 파일)
    done = ckpt.get("epoch", -1) + 1                                  # 끝난 epoch 수. 학습이 끝나면 epoch이 -1이라 0이 된다
    if ckpt.get("epoch", -1) == -1 or ckpt.get("optimizer") is None or done >= epochs:
        return "trained", epochs
    return "partial", done


state_rows = []
for i, (v, e) in enumerate(PLAN, start=1):
    state, done = run_state(v, e)
    state_rows.append({"순서": i, "run": run_name(v, e), "상태": STATE_TEXT[state], "끝난 epoch": f"{done}/{e}"})
display(pd.DataFrame(state_rows).set_index("순서"))

### 5-2. 순서대로 학습하고 채점

- 5-1의 순서대로 run을 하나씩 처리한다. 상태에 따라 새로 학습하거나 이어서 학습하고, 학습이 끝나면 바로 3-1의 `score_run`으로 채점한다(점수 3개, `scores.json`, W&B Summary).
    - 이어서 학습은 `last.pt`에서 `resume=True`로 한다. 처음과 같은 설정, 같은 run 폴더로 학습하고, W&B도 같은 실행에 이어서 기록한다.
    - 채점은 가이드대로 best.pt로 한다. best.pt는 Ultralytics가 epoch마다 잰 mAP50-95가 가장 좋은 epoch의 가중치다.
- run 하나에서 오류가 나면 그 오류를 출력하고 다음 run으로 넘어간다. 밤새 한 run의 오류 때문에 나머지 run이 멈추지 않게 하기 위해서다. 끝에 오류가 난 run을 모아 보여 주고, 셀을 다시 실행하면 그 run을 다시 시도한다.
- 직접 멈추려면 ■(중단) 버튼을 누른다. 셀 전체가 멈추고, 진행 중이던 epoch만 잃는다. epoch가 끝날 때 val 채점과 저장을 하는 동안(1분 안팎)은 피하는 것이 안전하다.
- 다시 시작하기
    - ■로 중단했으면 반드시 커널을 다시 시작한다. 같은 커널에서 다시 실행하면 중단된 학습이 잡고 있던 GPU 메모리가 풀리지 않아 학습이 크게 느려진다(5-3 D).
    - 커널을 다시 시작했거나 재부팅했으면: 0번 → 1번 → 3-1 → 5-1 → 이 셀 순서로 실행한다. 3-1은 이 셀이 쓰는 `score_run`과 `SCORE_KEYS`를, 5-1은 `PLAN`과 `run_state`를 만든다.
- 진행 확인: 진행 막대의 `s/it`(batch 하나에 걸리는 초)는 4-1에서 약 2.0초였다. batch 크기가 같아서 방법과 상관없이 비슷해야 한다. 3~4초 이상으로 오르면 GPU 메모리가 넘치거나 다른 프로그램 때문에 느려진 것이다.
- 학습 중에는 VS Code를 닫거나, 절전 모드로 들어가거나, 재부팅하거나, GPU를 많이 쓰는 게임을 하지 않는다.
- 각 run의 시작, 끝, 걸린 시간을 출력한다. 끝에 run별 점수 3개를 표로 보여 준다.

In [ ]:
def resume_one(variant, epochs):
    """중간에 멈춘 run을 last.pt에서 이어서 학습한다. 설정과 run 폴더, W&B 실행은 처음 그대로 쓴다."""
    model = RTDETR(str(RUNS_EXP2 / run_name(variant, epochs) / "weights" / "last.pt"))
    model.add_callback("on_train_start", require_wandb)
    model.train(resume=True)


failed = []
for variant, epochs in PLAN:
    name = run_name(variant, epochs)
    state, done = run_state(variant, epochs)
    print(f"\n[{time.strftime('%m-%d %H:%M')}] {name}: {STATE_TEXT[state]} ({done}/{epochs} epoch)")
    if state == "scored":
        continue
    if state == "broken":
        print(f"  건너뜀: run 폴더는 있는데 last.pt가 없습니다. runs/exp2/{name} 폴더를 지우고 이 셀을 다시 실행하면 새로 학습합니다.")
        failed.append(name)
        continue
    start = time.perf_counter()
    try:
        if state == "new":
            train_one(variant, epochs)
        elif state == "partial":
            resume_one(variant, epochs)
        score_run(RUNS_EXP2 / name, variant)
    except Exception as e:  # 오류가 난 run은 기록하고 다음 run으로 넘어간다. 중단(■)은 Exception이 아니라서 셀 전체가 멈춘다
        print(f"  오류: {type(e).__name__}: {e}")
        failed.append(name)
    finally:
        torch.cuda.empty_cache()
    print(f"[{time.strftime('%m-%d %H:%M')}] {name}: {(time.perf_counter() - start) / 3600:.2f}시간")

score_rows = []
for variant, epochs in PLAN:
    path = RUNS_EXP2 / run_name(variant, epochs) / "scores.json"
    scores = json.loads(path.read_text(encoding="utf-8")) if path.exists() else {}
    score_rows.append({"run": run_name(variant, epochs), "방법": variant, "epoch": epochs, **{k: scores.get(k) for k in SCORE_KEYS}})
print("\nrun별 점수 (mAP@[0.75:0.95], 빈 칸은 아직 채점 전)")
display(pd.DataFrame(score_rows).set_index("run").round(4))
print("오류가 나거나 건너뛴 run:", failed if failed else "없음")

### 5-3. 학습 결과 정리

- 5-2에서 학습한 6개 run이 어떻게 학습됐는지 정리한다. 방법과 epoch별 점수 비교는 7번에서 3-1의 점수로 하고, 여기서는 학습 과정만 본다.
- 자료는 run 폴더의 `results.csv`다. Ultralytics가 epoch마다 학습 손실, val 손실, val mAP, 학습률, 시간을 적은 파일이다.
    - 이어서 학습한 run도 한 파일에 이어서 적힌다.
    - 중단되어 끝내지 못한 epoch는 적히지 않는다.
- 여기 나오는 mAP50-95는 Ultralytics가 val 1,000장으로 잰 값이다. 모델의 모든 클래스를 평균하고 IoU 0.5~0.95를 보므로 대회 지표와 다르다(1-3). 학습이 어떻게 진행됐는지 보는 데만 쓴다.
- A. run별 진행 표
    - 끝난 epoch 수, best epoch(mAP50-95가 가장 높은 epoch. best.pt가 이 epoch의 가중치다), best epoch와 마지막 epoch의 mAP50-95
    - 마지막 2 epoch 동안 오른 폭(마지막 epoch의 값 − 그 2 epoch 전의 값). 크면 더 학습했을 때 오를 여지가 있다는 뜻이라, 세 번째 epoch 수를 정할 때 참고한다.
    - 마지막 epoch는 모자이크를 끄므로(1-2), 그 효과도 이 폭에 섞여 있다.
- B. epoch별 val mAP50-95 그래프
    - 색은 방법, 선 모양은 5 epoch run(점선)과 10 epoch run(실선)이다.
    - 세로선은 모자이크를 끈 epoch다(5 epoch run은 5, 10 epoch run은 10).
    - 오른쪽은 0.95 이상을 확대한 것이다.
- C. train과 val의 손실 그래프
    - giou, cls, l1 손실을 따로 그린다. 위 줄은 5 epoch run, 아래 줄은 10 epoch run이다. 실선은 train, 점선은 val이고, 로그 눈금이다.
    - 표에는 세 손실의 합으로 val 손실이 가장 낮은 epoch와, 마지막 epoch의 train과 val 손실을 적는다.
    - train 손실은 모자이크 같은 증강을 한 사진으로 잰 값이고, val 손실은 증강하지 않은 사진으로 잰 값이라 크기를 그대로 비교하기 어렵다. val 손실이 다시 오르는지, train과 val의 차이가 epoch가 지나며 벌어지는지를 본다.
- D. 학습 시간: 기록용이다(아래 D 설명).
- 실행: 0번, 1번, 5-1을 실행한 뒤 A부터 차례로 실행한다. 학습이나 예측은 하지 않고 파일만 읽는다.

In [ ]:
# A. run별 진행 표
MAP_COL = "metrics/mAP50-95(B)"                    # Ultralytics가 잰 val mAP50-95
LOSS_PARTS = ["giou_loss", "cls_loss", "l1_loss"]  # RT-DETR의 손실 세 가지

curves = {}  # run 이름 → results.csv 표
for variant, epochs in PLAN:
    name = run_name(variant, epochs)
    df = pd.read_csv(RUNS_EXP2 / name / "results.csv")
    df.columns = df.columns.str.strip()
    need = ["epoch", "time", MAP_COL, *[f"{s}/{p}" for s in ("train", "val") for p in LOSS_PARTS]]
    assert all(c in df.columns for c in need), f"{name}: results.csv에 없는 열이 있습니다. 있는 열: {list(df.columns)}"
    assert df["epoch"].astype(int).tolist() == list(range(1, epochs + 1)), \
        f"{name}: results.csv의 epoch가 1~{epochs}이 아닙니다: {df['epoch'].tolist()}"
    curves[name] = df
print("results.csv 열:", list(curves[run_name(*PLAN[0])].columns))

progress_rows = []
for variant, epochs in PLAN:
    name = run_name(variant, epochs)
    m = curves[name].set_index("epoch")[MAP_COL]
    progress_rows.append({"run": name, "방법": variant, "끝난 epoch": len(m),
                          "best epoch": int(m.idxmax()), "best mAP50-95": m.max(), "마지막 epoch mAP50-95": m.iloc[-1],
                          "마지막 2 epoch 동안 오른 폭": m.iloc[-1] - m.iloc[-3]})
print("run별 진행 (val mAP50-95는 Ultralytics 값, 대회 지표와 다르다)")
display(pd.DataFrame(progress_rows).set_index("run").round(4))

In [ ]:
# B. epoch별 val mAP50-95 그래프
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from matplotlib import font_manager

KOREAN_FONTS = ["Malgun Gothic", "AppleGothic", "NanumGothic"]  # 1차 노트북과 같은 글꼴 설정
installed = {f.name for f in font_manager.fontManager.ttflist}
FONT = next((name for name in KOREAN_FONTS if name in installed), None)
assert FONT is not None, f"한글 글꼴을 찾지 못했습니다. {KOREAN_FONTS} 중 하나를 설치한 뒤 커널을 다시 시작하세요."
plt.rcParams["font.family"] = FONT
plt.rcParams["axes.unicode_minus"] = False  # 한글 글꼴에서 음수 기호가 깨지지 않도록

METHOD_COLORS = {"v2": "#2a78d6", "v3": "#eb6834", "v4": "#1baf7a"}  # 1차 노트북의 seed 색 (색약 구분 확인한 색)
EPOCH_STYLES = {5: "--", 10: "-"}                                     # 5 epoch run 점선, 10 epoch run 실선

fig, axes = plt.subplots(1, 2, figsize=(15, 5), layout="constrained")
for variant, epochs in PLAN:
    df = curves[run_name(variant, epochs)]
    for ax in axes:
        ax.plot(df["epoch"], df[MAP_COL], color=METHOD_COLORS[variant], ls=EPOCH_STYLES[epochs], lw=1.6,
                marker="o", ms=3.5, label=f"{variant}, {epochs} epoch")
axes[1].set_ylim(0.95, 1.0)
for ax, title in zip(axes, ["전체", "0.95 이상 확대"]):
    for x, text in [(5, "모자이크 끔\n(5 epoch run)"), (10, "모자이크 끔\n(10 epoch run)")]:
        ax.axvline(x, color="gray", lw=1, ls="-.")
        ax.text(x - 0.15, 0.08, text, transform=ax.get_xaxis_transform(), ha="right", fontsize=8, color="dimgray")
    ax.set_xticks(range(1, 11))
    ax.set_xlabel("epoch")
    ax.set_ylabel("val mAP50-95 (Ultralytics)")
    ax.set_title(title, fontsize=10)
    ax.grid(alpha=0.3)
axes[0].legend(fontsize=9, ncol=2, loc="center right")  # 선이 없는 자리. 아래쪽은 "모자이크 끔" 글자와 겹친다
fig.suptitle("epoch별 val mAP50-95 (색: 방법, 점선: 5 epoch run, 실선: 10 epoch run)", fontsize=11)
plt.show()

In [ ]:
# C. train과 val의 손실 그래프
fig, axes = plt.subplots(2, 3, figsize=(17, 9), layout="constrained")
for row, epochs in enumerate([5, 10]):
    for col, part in enumerate(LOSS_PARTS):
        ax = axes[row, col]
        for variant in ["v2", "v3", "v4"]:
            df = curves[run_name(variant, epochs)]
            ax.plot(df["epoch"], df[f"train/{part}"], color=METHOD_COLORS[variant], ls="-", lw=1.6, marker="o", ms=3,
                    label=f"{variant} train")
            ax.plot(df["epoch"], df[f"val/{part}"], color=METHOD_COLORS[variant], ls="--", lw=1.6, marker="o", ms=3,
                    label=f"{variant} val")
        ax.set_yscale("log")
        ax.yaxis.set_major_locator(mticker.LogLocator(subs=(1, 2, 5)))
        ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v:g}"))  # 로그 눈금을 일반 숫자로 적는다
        ax.yaxis.set_minor_formatter(mticker.NullFormatter())
        ax.axvline(epochs, color="gray", lw=1, ls="-.")
        ax.set_xticks(range(1, epochs + 1))
        ax.set_xlabel("epoch")
        ax.set_ylabel("손실 (로그 눈금)")
        ax.set_title(f"{epochs} epoch run: {part}", fontsize=10)
        ax.grid(alpha=0.3)
axes[0, 0].legend(fontsize=8, ncol=3)
fig.suptitle("train과 val의 손실 (실선: train, 점선: val, 색: 방법, 세로선: 모자이크를 끈 epoch)", fontsize=11)
plt.show()

loss_rows = []
for variant, epochs in PLAN:
    name = run_name(variant, epochs)
    df = curves[name]
    train_loss = df[[f"train/{p}" for p in LOSS_PARTS]].sum(axis=1)
    val_loss = df[[f"val/{p}" for p in LOSS_PARTS]].sum(axis=1)
    loss_rows.append({"run": name, "val 손실 최저 epoch": int(df.loc[val_loss.idxmin(), "epoch"]), "val 손실 최저값": val_loss.min(),
                      "마지막 train 손실": train_loss.iloc[-1], "마지막 val 손실": val_loss.iloc[-1]})
print("손실 (giou + cls + l1의 합)")
display(pd.DataFrame(loss_rows).set_index("run").round(4))

#### D. 학습 시간 (기록)

- run마다 걸린 학습 시간을 기록으로 남긴다. `results.csv`의 `time` 열(그 학습을 시작한 뒤 지난 초)로 계산한다.
    - epoch 하나의 시간 = 이번 줄의 time − 앞 줄의 time. 이어서 학습하면 그 epoch에서 time이 0부터 다시 세지므로, 그 줄은 time 값을 그대로 쓴다.
    - 다시 세진 epoch는 time만으로 찾을 수 없을 때가 있다. 새로 센 값이 앞 줄보다 클 수 있기 때문이다(예: 앞 줄이 23분, 다시 센 첫 epoch가 40분). 그래서 이어서 학습을 시작한 epoch를 코드의 `RESUMED_FROM`에 직접 적는다. time이 앞 줄보다 작아진 epoch가 여기에 빠져 있으면 멈춘다.
    - 들어가는 것: epoch마다 학습, val 채점, 가중치 저장
    - 빠지는 것: 학습 준비, 학습 끝에 best.pt로 val을 한 번 더 채점하는 시간, 3-1의 채점, 중단되어 끝내지 못한 epoch
    - "나눠서 학습한 횟수"는 이어서 학습한 횟수 + 1이다.
- 눈에 띄게 느려진 구간: 중단한 뒤 커널을 다시 시작하지 않고 이어서 학습한 경우
    - 무엇을 했나: 5-2를 ■로 중단한 뒤, 커널을 다시 시작하지 않고 같은 커널에서 5-2를 다시 실행했다. 당시 5-2 설명에 "중단만 하고 커널이 살아 있으면 이 셀을 바로 다시 실행한다"고 적혀 있었고, 그대로 따랐다. 이 안내가 잘못됐다.
    - 결과: 그 커널에서 끝낸 v3_e5의 3~5 epoch와 v4_e5의 1 epoch가 크게 느려졌다(아래 epoch별 시간 표). v4_e5의 2 epoch는 재부팅한 뒤에 이어서 학습해 끝냈다. 5-2 출력에서 본 값은 다음과 같다.

| | 평소 | 이 구간 |
|---|---|---|
| GPU_mem (PyTorch가 잡은 GPU 메모리) | 12.7~13G | 22.5G |
| batch 하나 (s/it) | 2초 안팎 | 18초 안팎 |
| val 채점 (1,000장) | 31초 | 1분 38초 |

- 원인
    - 셀을 중단하면 주피터가 마지막 오류 정보(traceback)를 보관한다. 이 정보가 중단된 학습의 변수(모델, optimizer, 계산 중이던 값)를 붙잡고 있어서, 그 GPU 메모리가 풀리지 않는다.
    - 5-2의 `torch.cuda.empty_cache()`는 아무 변수도 쓰지 않는 메모리만 풀 수 있어서, 이 메모리는 풀지 못한다.
    - 그 위에 새 학습이 메모리를 또 잡아서 합계가 22.5G가 됐다. 그래픽카드 전용 메모리는 12GB라, 넘친 10G 정도는 Windows가 일반 RAM(공유 GPU 메모리)으로 돌려썼다. 일반 RAM은 그래픽카드 메모리보다 훨씬 느리다.
    - 근거
        - 학습 설정, AMP 확인 결과, 데이터는 그대로인데, 이어서 학습한 첫 epoch부터 GPU_mem이 늘었다.
        - 학습뿐 아니라 val 채점도 느려졌다.
        - 0~4번 셀의 실행 번호가 중단 전과 같았다. 커널을 다시 시작하지 않았다는 뜻이다.
        - 재부팅한 뒤 같은 v4_e5를 이어서 학습하자 GPU_mem이 12.7~13G, batch 하나가 2.1~3.4초로 돌아왔다.
    - 남은 메모리를 직접 재서 확인하지는 않았다. 위 근거로 이것이 원인이라고 본다.
- 점수에는 영향이 없다고 본다. 메모리 위치만 달랐고 계산은 같기 때문이다.
- 이후: ■로 중단한 뒤에는 항상 커널을 다시 시작하고, 0번 → 1번 → 3-1 → 5-1 → 5-2 순서로 실행한다.

In [ ]:
# D. 학습 시간 (기록)
RESUMED_FROM = {  # 이어서 학습을 시작한 epoch. 이 epoch에서 results.csv의 time이 0부터 다시 세진다
    "v3_rtdetr-l_e5": [3],     # 5-2 출력 "Resuming training ... from epoch 3"
    "v4_rtdetr-l_e5": [2, 3],  # 2: time이 앞 줄보다 작아진 epoch, 3: 5-2 출력 "from epoch 3"
    "v2_rtdetr-l_e10": [2],    # 5-2 출력 "from epoch 2"
}

time_rows, epoch_minutes = [], {}
for variant, epochs in PLAN:
    name = run_name(variant, epochs)
    t = curves[name]["time"].tolist()
    restarts = set(RESUMED_FROM.get(name, []))
    dropped = {i + 1 for i in range(1, len(t)) if t[i] < t[i - 1]}  # time이 앞 줄보다 작아진 epoch
    assert dropped <= restarts, f"{name}: time이 다시 세진 epoch {sorted(dropped - restarts)}가 RESUMED_FROM에 없습니다."
    minutes = [(t[i] if i == 0 or i + 1 in restarts else t[i] - t[i - 1]) / 60 for i in range(len(t))]
    epoch_minutes[name] = minutes
    time_rows.append({"run": name, "epoch 수": epochs, "학습 시간 (시간)": sum(minutes) / 60,
                      "가장 빠른 epoch (분)": min(minutes), "가장 느린 epoch (분)": max(minutes),
                      "나눠서 학습한 횟수": 1 + len(restarts)})
print("run별 학습 시간 (준비, 마무리 채점, 중단된 epoch는 빠진다)")
display(pd.DataFrame(time_rows).set_index("run").round(2))

per_epoch = pd.DataFrame({name: pd.Series(m, index=range(1, len(m) + 1)) for name, m in epoch_minutes.items()}).T
per_epoch.columns = [f"{e} epoch" for e in per_epoch.columns]
print("epoch별 시간 (분, 학습 + val 채점 + 저장)")
display(per_epoch.round(1))

### 5번 결과

- 6개 run(방법 3개 × 5, 10 epoch)을 모두 끝까지 학습하고 채점했다. 오류가 난 run은 없다(5-2). 방법과 epoch별 점수 비교는 7번에서 한다.

학습 진행 (5-3 A, B. Ultralytics가 잰 val mAP50-95)

- 5 epoch run은 마지막 epoch까지 오르고 있었다. 마지막 2 epoch 동안 v2 +0.0138, v4 +0.0061, v3 +0.0036 올랐다.
- 10 epoch run은 끝에서 거의 평평했다. 마지막 2 epoch 동안 v2 +0.0010, v3 +0.0019, v4 −0.0007이다. v4는 8 epoch가 가장 높았고(0.9906), v2와 v3는 마지막 epoch가 가장 높았다.
- 5 epoch run과 10 epoch run 모두, 모든 epoch에서 v4가 가장 높았다.
- v2_e10은 5 epoch(0.971)와 9 epoch(0.982)에서 한 번씩 내려갔다가 다시 올랐다.
- 5 epoch run의 마지막 값은 같은 방법의 10 epoch run이 5~8 epoch쯤에 낸 값과 비슷했다. Ultralytics는 학습률을 run의 마지막 epoch까지 lr0의 1%로 줄이므로, 짧은 run일수록 같은 epoch에서 학습률이 낮아 더 빨리 마무리되는 것으로 보인다.
- 그래서 10 epoch run이 끝에서 평평한 데에는 학습률이 거의 다 줄어든 탓도 있다. 이것만으로 epoch를 더 늘려도 오르지 않는다고 단정할 수는 없다.

손실 (5-3 C)

- 모든 run에서 val 손실이 마지막 epoch까지 내려갔고, val 손실이 가장 낮은 epoch가 마지막 epoch였다. 10 epoch 안에서는 과적합 기미가 보이지 않는다.
- train 손실은 모자이크를 쓰는 동안 val 손실보다 높았고, 모자이크를 끈 마지막 epoch에 크게 내려갔다. 10 epoch run은 마지막 epoch에서 train과 val 손실이 거의 같았다(차이 0.007 이하).
- v2_e10의 val 손실(giou, l1)도 mAP50-95가 내려간 5, 9 epoch에 올랐다.
- 손실 크기는 train, val 모두 v4 < v3 < v2였다. 다만 방법마다 클래스 수가 달라서(118, 96, 75), 특히 cls 손실은 방법끼리 그대로 비교하기 어렵다.

학습 시간 (5-3 D)

| run | 학습 시간 | 나눠서 학습한 횟수 |
|---|---:|---:|
| v2_rtdetr-l_e5 | 1.66시간 | 1 |
| v3_rtdetr-l_e5 | 9.58시간 | 2 |
| v4_rtdetr-l_e5 | 5.53시간 | 3 |
| v2_rtdetr-l_e10 | 2.72시간 | 2 |
| v3_rtdetr-l_e10 | 2.67시간 | 1 |
| v4_rtdetr-l_e10 | 3.65시간 | 1 |
| 합계 | 25.81시간 | |

- 6개 run에 25.81시간이 걸렸다. 4-1로 예측한 약 14.6시간(5-1)의 1.8배 정도다. 학습 준비, 마무리 채점, 중단되어 끝내지 못한 epoch는 빠진 시간이다.
- 커널을 다시 시작하지 않아 느려진 4개 epoch(v3_e5의 3~5 epoch, v4_e5의 1 epoch)에만 약 12시간이 들었다. 같은 방법의 10 epoch run에서는 epoch 하나에 v3 13~19분, v4 18~26분이었다. 원인과 이후 지킬 것은 5-3 D에 적었다.

## 6. 결과 기록

- 가이드 7단계대로 `notebooks/modeling/jaeone-rtdetr-exp2/runs.csv`에 run마다 한 줄을 적는다. 값은 모두 run 폴더에서 읽는다.

| 열 | 어디서 |
|---|---|
| run, 방법, 모델 | run 이름 |
| val, kaggle, test_only | `scores.json` (소수 넷째 자리까지) |
| commit | `dataset_info.json`의 `git_commit` 앞 7자리 |
| 학습 설정 | `args.yaml`. 가이드의 팀 공통값 항목(optimizer, lr0, epochs, imgsz, batch, seed)과 1-1, 1-2에서 직접 정한 값(momentum, warmup_bias_lr, warmup_epochs, close_mosaic) |
| best.pt 위치 | 공유 드라이브 `pill data/jw_runs/<run 이름>/weights/best.pt` |

- 5-1의 6개 run으로 파일 전체를 새로 쓴다. 가이드는 "한 줄을 추가"지만, 이렇게 하면 다시 실행해도 같은 줄이 두 번 들어가지 않는다. 이 폴더는 이 노트북의 기록만 둔다.
- 시간 측정 run(`smoke_v2_rtdetr-l_e1`)은 넣지 않는다.
- Kaggle public 점수는 가이드의 형식에 없어서 넣지 않는다. 8-2에 적었다.
- `runs.csv`는 git으로 커밋하는 파일이다. 팀 CSV와 같게 UTF-8, 줄바꿈 `\n`으로 쓴다.
- run 폴더는 이 셀이 올리지 않는다. 가이드 7단계대로 `runs/exp2/<run 이름>/` 폴더를 통째로 공유 드라이브 `pill data/jw_runs/`에 올린다(Drive 웹에서 새로 만들기 → 폴더 업로드). 올려야 "best.pt 위치"가 맞다.
- 실행: 0번, 1번, 5-1을 실행한 뒤 실행한다. 파일만 읽고 쓴다.

In [ ]:
SCORE_KEYS = ["val", "kaggle", "test_only"]  # 3-1과 같다
TRAIN_KEYS = ["optimizer", "lr0", "momentum", "warmup_bias_lr", "epochs", "warmup_epochs", "close_mosaic",
              "imgsz", "batch", "seed"]      # args.yaml에서 옮겨 적을 학습 설정
DRIVE_RUNS = "pill data/jw_runs"             # run 폴더를 올릴 공유 드라이브 위치 (팀 공지)

record_rows = []
for variant, epochs in PLAN:
    name = run_name(variant, epochs)
    run_dir = RUNS_EXP2 / name
    scores = json.loads((run_dir / "scores.json").read_text(encoding="utf-8"))
    info = json.loads((run_dir / "dataset_info.json").read_text(encoding="utf-8"))
    args = yaml.safe_load((run_dir / "args.yaml").read_text(encoding="utf-8"))
    record_rows.append({"run": name, "방법": variant, "모델": MODEL_NAME,
                        **{k: round(scores[k], 4) for k in SCORE_KEYS},
                        "commit": (info.get("git_commit") or "")[:7],
                        "학습 설정": ", ".join(f"{k}={args[k]}" for k in TRAIN_KEYS),
                        "best.pt 위치": f"{DRIVE_RUNS}/{name}/weights/best.pt"})

runs_record = pd.DataFrame(record_rows)
RECORD_DIR.mkdir(parents=True, exist_ok=True)
runs_record.to_csv(RECORD_DIR / "runs.csv", index=False, encoding="utf-8", lineterminator="\n")
print("저장:", (RECORD_DIR / "runs.csv").as_posix(), f"({len(runs_record)}줄)")
with pd.option_context("display.max_colwidth", None):
    display(runs_record.set_index("run"))

## 7. 결과 비교

- 3-1의 점수 3개로 방법(v2, v3, v4)과 epoch(5, 10)를 비교한다. 8번의 Kaggle 점수는 쓰지 않는다.
- 비교의 기준은 `val`이다. `kaggle`은 1차와 비교할 때(3-2), `test_only`는 test에만 나오는 18종을 볼 때 쓴다.
- 방법끼리 train이 어떻게 다른가(`data/datasets/README.md`, `classes.csv`)
    - v2: 클래스별 200개 안팎을 목표로 고른 사진 7,129장. 대회 클래스 74종의 train 객체 수 중앙값은 Kaggle train에 있던 56종이 200개, test 전용 18종이 209.5개다.
    - v3: 대회 클래스가 든 사진 8,974장 전부. 74종의 train 객체 수가 v2보다 많다(중앙값 56종 319.5개, 18종 337.5개).
    - v4: v3의 사진 + 대회 밖 알약만 든 사진 2,232장 = 11,206장. 대회 밖 알약은 모두 `others` 한 클래스다.
    - 대회 클래스 74종의 train 객체 수는 v3와 v4가 같다. 그래서 v3와 v4의 차이는 대회 밖 알약을 다루는 방법(v3는 22개 클래스로 따로, v4는 `others` 하나로 묶음)과, 그 알약만 든 사진 2,232장에서 온다.
- 한계: 방법과 epoch마다 seed 0으로 한 번씩만 학습했다. 1차 모델은 seed만 바꿔도 `kaggle`이 0.008 정도 움직였다(3-2). 2차에서 seed에 따라 val이 얼마나 움직이는지는 재지 않았으므로, 차이가 작으면 seed 차이일 수 있다.

### 7-1. 점수 표

- 첫 번째 표: run별 점수 3개와 val 순위(6개 run 안에서, 1이 가장 높다).
    - 1차 seed 0을 기준 줄로 넣는다(3-2의 `exp1_rescore.json`). 1차 모델은 AI Hub 클래스를 배우지 않아서 `val`과 `test_only`가 낮게 나오므로, 1차와는 `kaggle`만 비교한다.
    - kaggle − 1차 seed 0: 같은 Kaggle 사진 47장에서 1차보다 얼마나 달라졌는지다.
- 두 번째 표: 방법마다 5 epoch, 10 epoch 점수와 그 차이(10 − 5). 양수면 10 epoch가 높다.
- 실행: 0번, 1번, 5-1을 실행한 뒤 실행한다. 파일만 읽는다.

In [ ]:
SCORE_KEYS = ["val", "kaggle", "test_only"]  # 3-1과 같다
METHODS = ["v2", "v3", "v4"]

run_scores = {run_name(v, e): json.loads((RUNS_EXP2 / run_name(v, e) / "scores.json").read_text(encoding="utf-8"))
              for v, e in PLAN}
exp1_rescore = json.loads((RUNS_EXP2 / "exp1_rescore.json").read_text(encoding="utf-8"))  # 3-2, seed 0~2
exp1 = next(r for r in exp1_rescore if r["seed"] == 0)

# 첫 번째 표: run별 점수
compare_table = pd.DataFrame([{"run": run_name(v, e), "방법": v, "epoch": e, **{k: run_scores[run_name(v, e)][k] for k in SCORE_KEYS}}
                              for v, e in PLAN]).set_index("run")
compare_table.loc["1차 seed 0 (rtdetr-l_e100_s0_fix)"] = {"방법": "1차", "epoch": 100, **{k: exp1[k] for k in SCORE_KEYS}}
exp2_names = [run_name(v, e) for v, e in PLAN]
compare_table["val 순위"] = compare_table.loc[exp2_names, "val"].rank(ascending=False, method="min").astype("Int64")  # 1차 줄은 빈 칸
compare_table["kaggle − 1차 seed 0"] = compare_table["kaggle"] - exp1["kaggle"]
print("run별 점수 (mAP@[0.75:0.95], 1차와는 kaggle만 비교한다)")
display(compare_table.round(4))

# 두 번째 표: 방법마다 5 epoch와 10 epoch
diff_rows = []
for v in METHODS:
    s5, s10 = run_scores[run_name(v, 5)], run_scores[run_name(v, 10)]
    diff_rows.append({(k, col): value for k in SCORE_KEYS
                      for col, value in [("5 epoch", s5[k]), ("10 epoch", s10[k]), ("10 − 5", s10[k] - s5[k])]})
diff_table = pd.DataFrame(diff_rows, index=pd.Index(METHODS, name="방법"))
diff_table.columns = pd.MultiIndex.from_tuples(diff_table.columns)
print("방법마다 5 epoch와 10 epoch (양수면 10 epoch가 높다)")
display(diff_table.round(4))

### 7-2. 그래프

- 점수 3개를 방법별로 그린다. 빈 원은 5 epoch, 채운 원은 10 epoch이고, 두 점을 선으로 잇는다. 10 epoch 점 옆의 숫자는 10 − 5 차이다.
- 점수 차이가 작아서 세로축을 0.965부터 그린다. 세 그래프의 세로축은 같다.
- `kaggle` 그래프의 회색 띠는 1차 seed 0~2의 `kaggle` 범위, 점선은 1차 seed 0이다(3-2).
- 실행: 7-1과 5-3의 B(한글 글꼴과 방법별 색)를 실행한 뒤 실행한다.

In [ ]:
from matplotlib.lines import Line2D

assert "METHOD_COLORS" in globals(), "5-3의 B를 먼저 실행하세요. 그 셀에서 한글 글꼴과 방법별 색을 정한다."
kaggle_range = (min(r["kaggle"] for r in exp1_rescore), max(r["kaggle"] for r in exp1_rescore))  # 1차 seed 0~2

fig, axes = plt.subplots(1, 3, figsize=(15, 5), sharey=True, layout="constrained")
for ax, key in zip(axes, SCORE_KEYS):
    if key == "kaggle":
        ax.axhspan(*kaggle_range, color="0.88", lw=0, label=f"1차 seed 0~2 범위 ({kaggle_range[0]:.4f}~{kaggle_range[1]:.4f})")
        ax.axhline(exp1["kaggle"], color="0.45", ls="--", lw=1, label=f"1차 seed 0 ({exp1['kaggle']:.4f})")
        ax.legend(fontsize=8, loc="lower right")
    for i, v in enumerate(METHODS):
        y5, y10 = run_scores[run_name(v, 5)][key], run_scores[run_name(v, 10)][key]
        color = METHOD_COLORS[v]
        ax.plot([i - 0.12, i + 0.12], [y5, y10], color=color, lw=1.5)
        ax.plot(i - 0.12, y5, "o", ms=8, mfc="white", mec=color, mew=1.8)
        ax.plot(i + 0.12, y10, "o", ms=8, color=color)
        ax.text(i + 0.2, y10, f"{y10 - y5:+.4f}", va="center", fontsize=8, color="0.25")
    ax.set_xticks(range(len(METHODS)), METHODS)
    ax.set_xlim(-0.5, len(METHODS) - 0.3)
    ax.set_ylim(0.965, 1.0)
    ax.set_title(key, fontsize=10)
    ax.set_xlabel("방법")
    ax.grid(axis="y", alpha=0.3)
axes[0].set_ylabel("mAP@[0.75:0.95]")
axes[0].legend(handles=[Line2D([], [], marker="o", ls="", ms=8, mfc="white", mec="0.3", mew=1.8, label="5 epoch"),
                        Line2D([], [], marker="o", ls="", ms=8, color="0.3", label="10 epoch")], fontsize=8, loc="lower right")
fig.suptitle("방법과 epoch별 점수 (빈 원: 5 epoch, 채운 원: 10 epoch, 숫자: 10 epoch - 5 epoch, 세로축은 0.965부터)", fontsize=11)  # 맑은 고딕에 없는 − 대신 -
plt.show()

### 7-3. 클래스별 AP

- `scores.json`의 클래스별 AP(대회 클래스 74종, val 1,000장)로 점수 차이가 어느 클래스에서 나는지 본다.
    - 클래스별 AP는 IoU 0.75~0.95에서 잰 AP의 평균이다. `val` 점수는 74종 AP의 평균이고, `test_only` 점수는 test 전용 18종 AP의 평균이다. 첫 번째 표에 3-1의 점수를 나란히 적어 같은지 확인한다.
    - 클래스 이름, 그룹, val 객체 수, train 객체 수는 `data/datasets/classes.csv`에서 읽는다(읽기만 한다).
- 첫 번째 표: run별로 그룹의 AP 평균을 낸다. 그룹은 Kaggle train에 있던 56종과 test 전용 18종이다(`classes.csv`의 `group`). AP가 0.95 미만인 클래스 수와 가장 낮은 AP도 적는다.
- 두 번째 표와 그래프: 10 epoch run 세 개에서 방법끼리 AP 차이(가장 높은 방법 − 가장 낮은 방법)가 큰 15종
    - 10 epoch run으로 보는 이유: 세 방법 모두 val이 10 epoch에서 같거나 높았다(7-1).
    - 방법별 AP와 함께 방법별 train 객체 수를 적는다. v3와 v4는 대회 클래스의 train 객체 수가 같다(7번 설명).
    - 그래프는 AP가 같거나 아주 가까운 방법의 점이 겹치지 않도록, 방법마다 점을 위아래로 조금 비켜 그린다.
    - val 객체 수는 클래스마다 18~114개다. 적은 클래스는 박스 몇 개로도 AP가 크게 움직일 수 있다. val 객체 수도 함께 본다.
- 세 번째 표: 10 epoch run 세 개의 AP 평균이 가장 낮은 10종. 어느 방법으로도 어려운 클래스다.
- 실행: 7-1과 5-3의 B를 실행한 뒤 실행한다. 파일만 읽는다.

In [ ]:
CLASSES_CSV = Path("data/datasets/classes.csv")  # 팀 클래스 목록 (손으로 고치지 않는다. 여기서는 읽기만 한다)
GROUP_TEXT = {"kaggle": "Kaggle train 56종", "test_only": "test 전용 18종"}
TOP_N = 15

classes = pd.read_csv(CLASSES_CSV, encoding="utf-8-sig").set_index("category_id")
comp = classes[classes["competition"] == 1]  # 대회 클래스 74종
ap = pd.DataFrame({name: {int(c): value for c, value in s["per_class"].items()} for name, s in run_scores.items()})
assert set(ap.index) == set(comp.index), "scores.json의 클래스가 classes.csv의 대회 클래스 74종과 다릅니다."

# 첫 번째 표: 그룹별 AP 평균
group_rows = []
for v, e in PLAN:
    name = run_name(v, e)
    a = ap[name]
    row = {"run": name, "74종 평균": a.mean(), "val (3-1)": run_scores[name]["val"]}
    for group, text in GROUP_TEXT.items():
        row[f"{text} 평균"] = a.loc[comp.index[comp["group"] == group]].mean()
    row["test_only (3-1)"] = run_scores[name]["test_only"]
    row["AP 0.95 미만 클래스 수"] = int((a < 0.95).sum())
    row["가장 낮은 AP"] = a.min()
    group_rows.append(row)
print("run별 그룹 AP 평균 (74종 평균 = val, test 전용 18종 평균 = test_only)")
display(pd.DataFrame(group_rows).set_index("run").round(4))

# 두 번째 표: 10 epoch run에서 방법끼리 AP 차이가 큰 클래스
ap_cols = [f"AP {v}" for v in METHODS]
per_class = pd.DataFrame({"이름": comp["name"], "그룹": comp["group"].map(GROUP_TEXT), "val 객체": comp["n_val"]})
for v in METHODS:
    per_class[f"AP {v}"] = ap[run_name(v, 10)]
per_class["AP 차이"] = per_class[ap_cols].max(axis=1) - per_class[ap_cols].min(axis=1)
per_class["가장 높은 방법"] = per_class[ap_cols].idxmax(axis=1).str.replace("AP ", "")
per_class["가장 낮은 방법"] = per_class[ap_cols].idxmin(axis=1).str.replace("AP ", "")
per_class["AP 평균"] = per_class[ap_cols].mean(axis=1)
for v in METHODS:
    per_class[f"train {v}"] = comp[f"n_train_{v}"].astype("Int64")
top = per_class.sort_values("AP 차이", ascending=False).head(TOP_N)
print(f"10 epoch run에서 방법끼리 AP 차이가 큰 {TOP_N}종 (train은 train 객체 수)")
display(top.drop(columns="AP 평균").round(4))

fig, ax = plt.subplots(figsize=(10, 0.42 * TOP_N + 1.6), layout="constrained")
for row_i, (category_id, r) in enumerate(top.iterrows()):
    values = [r[c] for c in ap_cols]
    ax.plot([min(values), max(values)], [row_i, row_i], color="0.75", lw=2, zorder=1)
    for v, dy in zip(METHODS, (-0.2, 0.0, 0.2)):  # 같은 값이면 점이 겹쳐 가려지므로 방법마다 위아래로 비킨다
        ax.plot(r[f"AP {v}"], row_i + dy, "o", ms=8, color=METHOD_COLORS[v], mec="white", mew=1, zorder=2,
                label=v if row_i == 0 else None)
labels = [f"{r['이름'][:16]}{' (test 전용)' if r['그룹'] == GROUP_TEXT['test_only'] else ''}" for _, r in top.iterrows()]
ax.set_yticks(range(len(top)), labels, fontsize=9)
ax.invert_yaxis()  # 차이가 가장 큰 클래스를 위로
ax.set_xlabel("클래스 AP (IoU 0.75~0.95 평균, 10 epoch run)")
ax.grid(axis="x", alpha=0.3)
ax.legend(title="방법", fontsize=9, loc="lower left")
ax.set_title(f"방법끼리 AP 차이가 큰 {TOP_N}종 (회색 선: 가장 낮은 방법 ~ 가장 높은 방법, 점은 방법마다 위아래로 비켜 그림)", fontsize=11)
plt.show()

# 세 번째 표: 세 방법 모두 어려운 클래스
print("10 epoch run 세 개의 AP 평균이 가장 낮은 10종")
display(per_class.sort_values("AP 평균").head(10).round(4))

### 7번 결과

점수 (7-1, 7-2)

- val 기준으로 v4가 5 epoch(0.9858)와 10 epoch(0.9902) 모두 가장 높았다. 6개 run 중 1위는 `v4_rtdetr-l_e10`이다.
- 세 방법 모두 10 epoch가 5 epoch보다 점수 3개가 같거나 높았다.
    - val이 오른 폭은 v3 +0.0097, v4 +0.0044, v2 +0.0009 순서다. v3는 5 epoch에서 val이 가장 낮았다가 10 epoch에서 2위가 됐다.
    - v2는 점수 3개 모두 0.001 미만으로 거의 변하지 않았다.
- 10 epoch에서 방법끼리 val 차이는 0.0020(v4 − v3)~0.0093(v4 − v2)으로 작다. seed를 바꿔 재지 않아서, 특히 v4와 v3의 차이는 seed 차이일 수 있다(7번 설명).
- `kaggle`(Kaggle 사진 47장)은 v3_e10(0.9971)과 v4_e10(0.9908)만 1차 seed 0~2의 범위(0.9782~0.9866)보다 높았고, 나머지 4개 run은 그 범위 안에 있다. 1차 모델도 이 47장에서 이미 0.98 안팎이라, AI Hub 데이터를 더한 효과는 이 47장으로는 크게 드러나지 않는다.
- `test_only`는 1차 모델이 0이었고(3-2), 2차는 0.9722~0.9875다. 2차 모델은 1차가 배우지 못한 test 전용 18종을 AI Hub 사진으로 배웠다. 다만 이 점수는 val의 AI Hub 사진으로 잰 것이다.

클래스별 AP (7-3)

- 74종 평균은 `val`과, test 전용 18종 평균은 `test_only`와 6개 run 모두 같았다. 클래스별 AP와 점수가 같은 계산에서 나왔다.
- 모든 run에서 Kaggle train 56종의 평균이 test 전용 18종보다 높았다(차이 0.0023~0.0083).
- AP가 0.95 미만인 클래스는 v4_e10이 3종으로 가장 적었다(v3_e10 5종, v2_e10 10종).
- 10 epoch run에서 방법끼리 AP 차이가 큰 15종
    - 가장 낮은 방법이 v2인 클래스가 11종이고, 그중 10종은 v2의 train 객체 수가 v3, v4보다 적었다(v2는 클래스별 200개 안팎으로 맞췄다). train 객체가 적은 것과 AP가 낮은 것이 함께 보이지만, 그것이 원인인지는 확인하지 않았다.
    - 가장 높은 방법은 v4가 9종, v3가 6종이다. 뉴로메드정은 v3와 v4가 모두 1.0이라 표에서 v3로 셌다.
    - 보령부스파정(1900), 카발린캡슐(41768), 뮤테란캡슐(2483)은 세 방법의 train 객체 수가 같은데도 AP가 0.026~0.065 차이 났다. 대회 클래스의 학습 양이 같아도, 함께 학습한 다른 사진이나 seed에 따라 클래스 AP가 이만큼 움직일 수 있다는 뜻이다.
    - 15종 중 11종은 val 객체가 30개 이하다. 객체가 적으면 박스 몇 개로도 AP가 크게 움직이므로, 클래스 하나의 차이보다 여러 클래스에 걸친 경향으로 본다.
    - test 전용 18종 중 4종(메가파워정, 브린텔릭스정, 케이캡정, 비타비백정)이 15종에 들었다.
- 세 방법 모두 어려운 10종(AP 평균 0.942~0.957) 중 8종은 Kaggle train 56종, 2종은 test 전용(넥시움정, 삐콤씨에프정)이다.

정리

- 팀 기준(val)으로 고르면 방법은 v4, epoch는 10이다(`v4_rtdetr-l_e10`). v4는 두 epoch 모두, 두 클래스 그룹 모두에서 가장 높았고, AP가 0.95 미만인 클래스도 가장 적었다.
- 다만 방법 사이 차이가 작고 한 번씩만 학습했다. v4와 v3의 순위는 seed를 바꿔 다시 재야 확실해진다.

## 8. 제출

- 가이드 8단계대로 val로 고른 run만 Kaggle에 제출한다(하루 10회).
- 고르는 기준: 방법마다 5 epoch와 10 epoch 중 `val`이 높은 run을 고른다.
    - `val`은 1,000장으로 잰 점수라 세 점수 중 사진이 가장 많고, 가이드가 방법과 epoch를 비교하는 기준으로 정한 점수다.
    - `kaggle`은 47장이라 흔들림이 크다. 1차 모델은 seed만 바꿔도 0.008 정도 움직였다(3-2).
    - `test_only`는 test에만 나오는 18종만 본다.
    - Kaggle 점수로 고르면 test에 맞춰 고르는 셈이 된다. 그래서 Kaggle 점수는 고르는 데 쓰지 않고, 고른 run의 결과를 확인하는 데만 쓴다.

| 방법 | val 5 epoch | val 10 epoch | 고른 run |
|---|---:|---:|---|
| v2 | 0.9800 | 0.9809 | `v2_rtdetr-l_e10` |
| v3 | 0.9785 | 0.9882 | `v3_rtdetr-l_e10` |
| v4 | 0.9858 | 0.9902 | `v4_rtdetr-l_e10` |

- 세 방법 모두 10 epoch다. `kaggle`과 `test_only`도 10 epoch가 같거나 높다(5-2 표).
- v2는 차이가 0.0009로 작다. 그래도 10 epoch로 고르면 세 방법의 epoch 수가 같아서, Kaggle에서 방법끼리 비교할 때 epoch 차이가 섞이지 않는다.
- 세 run을 제출하면 val의 방법 순위(v4, v3, v2)가 Kaggle에서도 같은지 볼 수 있다. 1차는 val과 Kaggle 점수의 차이가 약 0.56이었다(1차 노트북 5번 결과).
- 5 epoch run 두 개를 더 제출한다. 팀원들의 실험에서 5 epoch run의 Kaggle 점수가 더 높게 나왔다고 해서, RT-DETR에서도 그런지 확인하려는 것이다. val로는 세 방법 모두 10 epoch가 같거나 높았으므로(위 표), Kaggle에서 5 epoch가 높게 나오면 val과 Kaggle이 epoch 수에서도 어긋난다는 뜻이 된다. 이 제출도 run을 고르는 데는 쓰지 않는다.
    - `v4_rtdetr-l_e5`: 5 epoch run 중 val이 가장 높다. 이미 제출한 `v4_rtdetr-l_e10`과 epoch 수만 다르다.
    - `v3_rtdetr-l_e5`: 5 → 10 epoch에서 val이 가장 많이 올랐다(+0.0097). val로는 10 epoch가 가장 분명하게 나은 방법이라, 여기서도 Kaggle이 5 epoch를 더 높게 매기면 어긋남의 근거가 가장 강하다.
    - v2는 5 epoch와 10 epoch의 val 차이가 0.0009로 작아서, 어느 쪽이 높게 나와도 val과 어긋난다고 보기 어려워 뺐다.
    - 같은 방법의 5 epoch와 10 epoch의 Kaggle 점수 차이가 0.004보다 작으면 차이가 없다고 본다. 1차 모델은 seed만 바꿔도 Kaggle 점수가 0.004 정도 움직였다(1차 노트북 5-3).

### 8-1. 제출 CSV 만들기

- 고른 run의 best.pt로 test 사진을 예측하고, 가이드 8단계의 팀 함수로 제출 CSV를 쓴다.
    - 예측 조건: 입력 크기 640, 점수 하한 0.01. 가이드 8단계의 값이고, 1차 제출과 같다.
    - `predictions_from_ultralytics`로 예측을 바꾸고 `write_submission`으로 쓴다. 대회 클래스 74개는 학습 폴더의 `classes.csv`에서 `competition_from_classes`로 읽는다.
    - 대회 클래스 밖의 예측(v4의 `others`, v2와 v3의 AI Hub에만 있는 클래스)은 `write_submission`이 뺀다.
    - 열은 `annotation_id, image_id, category_id, bbox_x, bbox_y, bbox_w, bbox_h, score`다. `image_id`는 test 파일 번호(`12.png` → 12)이고, bbox는 원본 사진 픽셀의 x, y, 너비, 높이다.
- 1차 제출과 다른 점: 1차는 사진 밖으로 나간 박스를 사진 안으로 잘라서 냈다. 팀 함수는 자르지 않는다. 2차 val 채점(3-1)도 자르지 않으므로, val 점수와 같은 조건이 되도록 팀 함수를 그대로 쓴다.
- 저장: `submissions/exp2_<run 이름>.csv`. 이 폴더는 git이 무시한다. `git add -f`로 올리지 않는다(가이드 "하지 말 것").
- 같은 이름의 CSV가 이미 있으면 다시 예측하지 않고, 그 파일을 읽어서 확인만 한다.
    - 같은 모델이라도 다시 예측하면 점수 끝자리가 달라질 수 있다. Kaggle에 올린 파일과 어긋나지 않게 하려는 것이다(1차 노트북 5번과 같은 방식).
    - 다시 만들려면 `REPREDICT = True`로 바꾼다. 이때는 있던 파일을 덮어쓴다.
- 확인: run마다 CSV를 다시 읽어 아래를 보고 표로 보여 준다.
    - 형식: 열 이름과 순서, `annotation_id`가 1부터 빠짐없이 매겨졌는지, `image_id`가 모두 test 파일 번호인지, `category_id`가 모두 대회 클래스인지. 하나라도 어긋나면 멈춘다.
    - 모양: 줄 수, 예측이 하나라도 있는 test 사진 수, 사진당 줄 수(평균, 최대), 가장 낮은 점수, 너비나 높이가 0 이하인 박스 수
    - 대회 클래스 밖이라 뺀 예측 수. 새로 예측할 때만 알 수 있어서, 저장된 CSV를 읽을 때는 빈 칸이다.
- 제출은 Kaggle 웹에서 CSV를 직접 올린다. 제출 설명(description)에 run 이름을 적어 두면 8-2에서 점수를 옮길 때 헷갈리지 않는다.
- 실행: 0번, 1번을 실행한 뒤 실행한다. 새로 예측하면 run마다 몇 분 걸린다(재 보지 않은 추정).

In [ ]:
from pill_detection.evaluate import SUBMISSION_COLUMNS, competition_from_classes, write_submission

SUBMIT_RUNS = [("v2", 10), ("v3", 10), ("v4", 10),  # 방법마다 val이 높은 epoch (8번 표)
               ("v3", 5), ("v4", 5)]                # 5 epoch 확인용 (8번 설명)
TEST_DIR = Path("data/raw/test_images")             # Kaggle test 사진 (가이드 8단계)
SUBMIT_DIR = Path("submissions")                     # git이 무시하는 폴더
SUBMIT_CONF = 0.01                                   # 제출용 예측의 점수 하한 (가이드 8단계, 1차 제출과 같다)
REPREDICT = False                                    # True: 이미 있는 CSV도 다시 예측해서 덮어쓴다

test_paths = sorted(TEST_DIR.glob("*.png"))
assert test_paths, f"test 사진이 없습니다: {TEST_DIR.as_posix()}. EXPERIMENTS.md 3단계대로 test 사진을 준비하세요."
assert all(p.stem.isdigit() for p in test_paths), "파일 이름이 숫자가 아닌 test 사진이 있습니다."
TEST_IDS = {int(p.stem) for p in test_paths}
print("test 사진:", len(TEST_IDS), "장")


def submission_path(variant, epochs):
    """제출 CSV 경로. 예: submissions/exp2_v2_rtdetr-l_e10.csv"""
    return SUBMIT_DIR / f"exp2_{run_name(variant, epochs)}.csv"


def make_submission(variant, epochs):
    """run의 best.pt로 test 사진을 예측해 제출 CSV를 쓴다. 대회 클래스 밖이라 뺀 예측 수를 돌려준다."""
    weights = RUNS_EXP2 / run_name(variant, epochs) / "weights" / "best.pt"
    assert weights.exists(), f"best.pt가 없습니다: {weights.as_posix()}. 5-2에서 학습이 끝난 run인지 확인하세요."
    model = RTDETR(str(weights))
    results = model.predict(source=str(TEST_DIR), imgsz=COMMON["imgsz"], conf=SUBMIT_CONF, device=DEVICE,
                            stream=True, verbose=False)
    predictions = predictions_from_ultralytics(results)
    n_rows = write_submission(predictions, submission_path(variant, epochs),
                              competition_from_classes(EXP2_DIR / variant / "classes.csv"))
    return len(predictions) - n_rows


sub_rows = []
for variant, epochs in SUBMIT_RUNS:
    name = run_name(variant, epochs)
    path = submission_path(variant, epochs)
    dropped = None  # 저장된 CSV를 읽으면 알 수 없다
    if path.exists() and not REPREDICT:
        print(f"{name}: 저장된 CSV를 읽음 ← {path.as_posix()}")
    else:
        start = time.perf_counter()
        dropped = make_submission(variant, epochs)
        torch.cuda.empty_cache()
        print(f"{name}: 새로 예측해서 저장 → {path.as_posix()} ({(time.perf_counter() - start) / 60:.1f}분)")

    # 형식 확인: 하나라도 어긋나면 멈춘다
    sub = pd.read_csv(path)
    competition = competition_from_classes(EXP2_DIR / variant / "classes.csv")
    assert list(sub.columns) == SUBMISSION_COLUMNS, f"{name}: 열이 제출 형식과 다릅니다: {list(sub.columns)}"
    assert sub["annotation_id"].tolist() == list(range(1, len(sub) + 1)), f"{name}: annotation_id가 1부터 차례대로가 아닙니다."
    assert set(sub["image_id"]) <= TEST_IDS, f"{name}: test 파일 번호가 아닌 image_id가 있습니다."
    assert set(sub["category_id"]) <= competition, f"{name}: 대회 클래스가 아닌 category_id가 있습니다."

    # 모양 확인
    per_image = sub.groupby("image_id").size()
    scores = json.loads((RUNS_EXP2 / name / "scores.json").read_text(encoding="utf-8"))
    sub_rows.append({"run": name, "val": scores["val"], "줄 수": len(sub),
                     "예측이 있는 사진": f"{len(per_image)}/{len(TEST_IDS)}",
                     "사진당 줄 (평균)": round(float(per_image.mean()), 1), "사진당 줄 (최대)": int(per_image.max()),
                     "가장 낮은 점수": float(sub["score"].min()),
                     "너비나 높이가 0 이하": int(((sub["bbox_w"] <= 0) | (sub["bbox_h"] <= 0)).sum()),
                     "대회 밖이라 뺀 예측": dropped})

print("\n제출 CSV 확인 (Kaggle에는 이 파일들을 올린다)")
display(pd.DataFrame(sub_rows).set_index("run").round(4))

### 8-2. Kaggle 점수

- 8-1의 CSV 5개(10 epoch run 3개, 5 epoch run 2개)를 Kaggle에 올려 받은 public 점수다(2026-11-02 제출, 제출 목록 화면의 Public Score). 점수를 셀 코드에 직접 적는다. 커밋할 때 출력은 지워지지만 코드는 남는다.
- Kaggle public 점수는 test의 일부로 매긴 점수일 수 있다. 몇 %로 매기는지는 확인하지 않았다(1차 노트북 5-3과 같다).
- 첫 번째 표: run별 val, kaggle, test_only(`scores.json`)와 public 점수를 나란히 놓는다.
    - 기준 줄로 1차 seed 0을 넣는다. 점수 3개는 3-2에서 저장한 `runs/exp2/exp1_rescore.json`의 값이고, public 0.42152는 1차 노트북 5-3의 값이다.
    - val 순위, public 순위: 2차 run 5개 안에서 매긴 순위다(1이 가장 높다).
    - val − public: 2차 run만 계산한다. 1차 모델은 2차 val의 AI Hub 클래스를 배우지 않아서 val이 낮게 나오므로(3-2) 빈 칸으로 둔다.
    - kaggle − public: 모든 줄을 계산한다. `kaggle`은 test와 같은 Kaggle 사진 47장의 점수이고, 1차와 2차를 같은 사진과 같은 채점 함수로 쟀다. 그래서 1차와 2차의 차이를 나란히 볼 수 있다.
    - 1차보다 오른 public: 1차 seed 0의 public 점수와의 차이다.
- 두 번째 표: 5 epoch run도 제출한 방법(v3, v4)에서 5 epoch와 10 epoch를 나란히 놓는다(8번 설명).
    - val 차이와 public 차이는 모두 "10 epoch − 5 epoch"다. 양수면 10 epoch가 높다.
    - public 판단: 차이가 0.004보다 작으면 "차이 없음"으로 본다. 8번 설명에서 제출 전에 정한 기준이다.
- 1차 제출은 사진 밖으로 나간 박스를 잘랐고, 2차 제출은 자르지 않았다(8-1). 1차와의 차이에 이 영향도 조금 섞여 있을 수 있다.
- 실행: 0번, 1번을 실행한 뒤 실행한다. 학습이나 예측은 하지 않는다.

In [ ]:
KAGGLE_PUBLIC = {  # Kaggle 제출 목록의 Public Score (2026-11-02 제출, 제출 설명에 run 이름)
    "v2_rtdetr-l_e10": 0.61516,
    "v3_rtdetr-l_e10": 0.60655,
    "v4_rtdetr-l_e10": 0.61140,
    "v3_rtdetr-l_e5": 0.60295,   # 5 epoch 확인용 (8번 설명)
    "v4_rtdetr-l_e5": 0.61558,
}
EXP1_NAME = "1차 seed 0 (rtdetr-l_e100_s0_fix)"
EXP1_PUBLIC = 0.42152  # 1차 노트북 5-3
SCORE_KEYS = ["val", "kaggle", "test_only"]  # 3-1과 같다

exp1 = next(r for r in json.loads((RUNS_EXP2 / "exp1_rescore.json").read_text(encoding="utf-8")) if r["seed"] == 0)
public_rows = [{"run": EXP1_NAME, **{k: exp1[k] for k in SCORE_KEYS}, "Kaggle public": EXP1_PUBLIC}]
for name, public in KAGGLE_PUBLIC.items():
    scores = json.loads((RUNS_EXP2 / name / "scores.json").read_text(encoding="utf-8"))
    public_rows.append({"run": name, **{k: scores[k] for k in SCORE_KEYS}, "Kaggle public": public})
public_table = pd.DataFrame(public_rows).set_index("run")

exp2_runs = public_table.loc[list(KAGGLE_PUBLIC)]
public_table["val 순위"] = exp2_runs["val"].rank(ascending=False, method="min").astype("Int64")
public_table["public 순위"] = exp2_runs["Kaggle public"].rank(ascending=False, method="min").astype("Int64")
public_table["val − public"] = exp2_runs["val"] - exp2_runs["Kaggle public"]  # 1차 줄은 빈 칸
public_table["kaggle − public"] = public_table["kaggle"] - public_table["Kaggle public"]
public_table["1차보다 오른 public"] = public_table["Kaggle public"] - EXP1_PUBLIC

print(f"2차 run {len(KAGGLE_PUBLIC)}개와 1차 seed 0의 점수 (모두 mAP@[0.75:0.95])")
display(public_table.round(4))

# 같은 방법의 5 epoch와 10 epoch
EPOCH_SAME_LIMIT = 0.004  # 이보다 작은 public 차이는 "차이 없음" (8번 설명: 1차 모델은 seed만 바꿔도 이만큼 움직였다)
pair_rows = []
for variant in ["v2", "v3", "v4"]:
    e5, e10 = run_name(variant, 5), run_name(variant, 10)
    if e5 not in KAGGLE_PUBLIC or e10 not in KAGGLE_PUBLIC:
        continue  # 두 run을 모두 제출한 방법만
    diff = KAGGLE_PUBLIC[e10] - KAGGLE_PUBLIC[e5]
    pair_rows.append({"방법": variant, "val 5 epoch": public_table.loc[e5, "val"], "val 10 epoch": public_table.loc[e10, "val"],
                      "val 차이 (10 − 5)": public_table.loc[e10, "val"] - public_table.loc[e5, "val"],
                      "public 5 epoch": KAGGLE_PUBLIC[e5], "public 10 epoch": KAGGLE_PUBLIC[e10], "public 차이 (10 − 5)": diff,
                      "public 판단": "차이 없음" if abs(diff) < EPOCH_SAME_LIMIT else ("10 epoch가 높음" if diff > 0 else "5 epoch가 높음")})
print("같은 방법의 5 epoch와 10 epoch")
display(pd.DataFrame(pair_rows).set_index("방법").round(4))

## 9. test 예측 확인

### 왜 하는가: val과 Kaggle 점수의 차이

- val은 0.98~0.99인데 Kaggle public은 0.60~0.62다(8-2). 1차(약 0.56)보다는 줄었지만 0.37 안팎의 차이가 남았다.
- test 정답은 Kaggle에만 있고, Kaggle은 점수 하나만 알려 준다. 그래서 어느 예측이 틀렸는지는 알 수 없다.
- 이 섹션은 제출한 예측(best.pt가 test 사진에 낸 답)을 test 사진 위에 그려 보고, 틀렸을 가능성이 큰 곳을 눈으로 확인한다.
- 먼저 차이를 만들 수 있는 이유를 정리한다. 아래는 모두 확인하지 않은 후보다.

먼저 알아 둘 점수 계산

- 대회 점수는 클래스마다 AP를 내서 평균한다. test에는 대회 클래스 74종이 모두 있다(`classes.csv`의 `n_test`, 객체 3,229개).
- 객체가 적은 클래스도 많은 클래스와 똑같이 평균에 들어간다. test 전용 18종은 객체로는 14%(453개)지만, 점수로는 74종 중 18종, 24%를 차지한다.
- 1차 점수로 거꾸로 계산해 볼 수 있다. 1차 모델은 18종을 배우지 않아 그 AP가 0이었다.
    - 0.4215 = (56종의 AP 평균 × 56 + 0 × 18) ÷ 74이므로, 1차에서 56종의 test AP 평균은 약 0.56이다.
    - 같은 56종이 val의 대회 사진 47장에서는 0.98이었다. 배운 56종도 test에서는 크게 낮았다는 뜻이다.
    - Kaggle public이 74종을 모두 채점한다고 가정한 계산이다. public이 test 일부로 매겨진다면 숫자가 조금 다를 수 있다.

차이를 만들 수 있는 이유 (후보)

| 후보 | 내용 | 지금까지의 근거 | 9번에서 볼 수 있나 |
|---|---|---|---|
| 1. 대회 사진 속 18종 | 18종은 AI Hub 사진으로만 배웠다. val에도 대회 사진 속 18종은 없어서, 대회 사진에서 18종을 얼마나 맞히는지 잴 수 없다 | 18종을 배우자 1차보다 0.19 올랐다. 18종이 점수의 24%를 차지한다 | 볼 수 있다. 18종으로 예측한 알약, 18종이 다른 약으로 보인 알약 |
| 2. 비슷한 약끼리 헷갈림 | test 사진에는 56종과 18종이 섞여 있다. 비슷한 약을 높은 점수로 잘못 내면 두 약의 AP가 함께 깎인다 | 1차 5-6에서 배우지 않은 약을 닮은 배운 약 번호로 점수 0.54~0.91에 냈다 | 볼 수 있다. 한 알약에 약 번호 둘, 한 사진에 같은 약 두 번 |
| 3. val의 대회 사진이 일부 약만 다룸 | val의 대회 사진 47장에는 56종 중 30종만 나온다(1차 노트북 2-3). 나머지 26종과 18종은 val에서 AI Hub 사진으로만 잰다 | 1차에서 56종의 test AP가 약 0.56으로, val(0.98)보다 크게 낮았다 | 일부. 56종 쪽 신호를 따로 센다 |
| 4. 정답 박스의 차이 | IoU 0.75~0.95는 박스가 조금만 어긋나도 깎는다. test 정답 박스가 우리가 배운 박스와 다르게 그려졌다면 모든 클래스가 깎인다 | 반대 근거가 있다. 대회 정답 박스를 쓴 대회 사진 47장의 `kaggle` 점수가 0.98~0.997이다. 팀이 고친 대회 train 박스는 21개(약 3%)다 | 볼 수 없다. test 정답 박스가 없다 |
| 5. 점수 자체의 흔들림 | public 점수가 test 일부로 매겨질 수 있고, test 객체가 3~6개뿐인 클래스도 있다 | 1차는 seed만 바꿔도 0.004 움직였다 | 볼 수 없다 |

- 이 섹션에서는 1, 2번, 곧 약을 헷갈리거나 못 찾는 경우를 본다. 4번은 test 정답 박스가 없어서 볼 수 없다.

### 진행

- 기준은 v4다. v4는 val에서 5, 10 epoch 모두 1위였고(7-1), Kaggle에서도 상위였다(8-2). 같은 방법의 `v4_rtdetr-l_e10`과 `v4_rtdetr-l_e5`를 나란히 본다. epoch만 다른 두 run이 다르게 답한 알약은 모델이 특히 불안정한 곳이다.
- 예측은 8-1에서 만들어 Kaggle에 올린 제출 CSV를 그대로 읽는다.
- test에는 정답이 없어서 "틀린 예측"을 확정할 수는 없다. 정답 없이도 알 수 있는 모순(아래 9-2)으로 틀렸을 가능성이 큰 예측을 고르고, 사진으로 확인한다.
    - 팀 데이터 목록(`images.csv`)의 사진 12,206장은 모두 알약이 3개나 4개이고, 한 사진에 같은 약이 두 번 들어간 경우가 없다. test 사진도 같은 방식으로 만들었다고 보고 이를 기준으로 쓴다.

| 단계 | 내용 |
|---|---|
| 9-1 | 두 run의 점수 0.5 이상 예측을 알약 자리로 묶는다(같은 사진에서 IoU 0.5 이상 겹치면 같은 알약) |
| 9-2 | 틀렸을 가능성이 큰 신호를 센다. 한 자리에 약 번호 둘 이상, 한 사진에 같은 약 두 번, 자리 5개 이상, 2개 이하, 두 run이 다른 약으로 본 자리. 약 번호별로 모아 18종과 56종 중 어디에 몰리는지 본다 |
| 9-3 | (9-2 결과를 보고 넣은 단계) 진단 제출: 점수 0.5 이상 답만 남긴 CSV로 Kaggle 채점 방식을 확인한다 |
| 9-4 | 9-2 결과를 보고 사례를 골라, test 사진의 예측과 val 정답 알약을 나란히 놓고 눈으로 확인한다 |

### 9-1. 알약 자리로 묶기

`others` 답이 빠진 이유와 다시 예측하는 이유

- v4는 대회 클래스 74종 밖의 알약(AI Hub에만 있는 44종)을 모두 `others` 한 클래스로 묶어 배웠다. v4 train 객체 42,800개 중 16,552개(39%)가 `others`다(`images.csv`). 그래서 v4는 알약을 보고 "74종 중 하나" 또는 "`others`(대회 밖 알약)"라고 답한다.
- 제출 CSV에는 대회 클래스 74종의 답만 들어간다. Kaggle 정답에는 `others`가 없고, test에는 대회 밖 알약이 없기 때문이다(`evaluate.py` 설명). 그래서 8-1의 `write_submission`이 `others` 답을 지웠다. v4_e10은 점수 0.01 이상인 대회 밖 답 6,093개가 지워졌다(8-1).
- 지워진 답이 문제가 되는 경우
    - test 알약을 `others`로 자신 있게 보고, 정작 맞는 약 번호는 낮은 점수로만 냈다면, 제출 CSV에는 낮은 점수의 답만 남는다.
    - Kaggle에서는 그 알약을 못 찾았거나 낮은 점수로 찾은 것으로 채점된다.
    - test에는 대회 밖 알약이 없으므로, test 알약 위에 자신 있게 낸 `others`는 모두 틀린 답이다.
- val은 다르다.
    - val의 AI Hub 사진에는 대회 밖 알약이 737개 있다(사진 737장에 하나씩). 그 알약에는 `others`가 맞는 답이다.
    - 3-1의 채점은 대회 밖 알약과 `others` 답을 빼고 74종만 채점한다. val 정답에서 빠진 것은 없고, val 점수는 `others` 답 때문에 깎이지 않는다.
- `others` 답은 제출 CSV에 없으므로, 두 run의 best.pt로 test 사진을 다시 예측해서 `others` 답만 꺼낸다.
    - 예측 조건은 8-1과 같다(입력 640, 점수 0.01 이상). 분석용이고, 제출 CSV는 바꾸지 않는다. run마다 20초 안팎이다(8-1에서 잰 시간).
    - 같은 모델이라도 GPU 계산 순서 때문에 점수 끝자리가 조금 다를 수 있다. 다시 예측한 대회 클래스 답의 수를 제출 CSV의 줄 수와 나란히 적어, 같은 모델의 답인지 확인한다.

알약 자리로 묶기

- 두 run마다 제출 CSV의 답과 다시 예측한 `others` 답 중 점수 0.5 이상인 것만 남긴다. 0.5는 "모델이 자신 있게 낸 답"을 세려는 기준으로, 1차 5-2와 같다.
- 같은 사진에서 박스끼리 IoU 0.5 이상 겹치면 같은 알약으로 보고 "자리" 하나로 묶는다(1차 5-4와 같은 기준).
    - 묶는 순서: 점수가 높은 박스부터 본다. 이미 있는 자리의 대표 박스(그 자리에서 점수가 가장 높은 박스)와 IoU 0.5 이상이면 그 자리에 넣고, 아니면 새 자리를 만든다.
    - 자리마다 대표 약 번호와 점수, 대표 박스, 그 자리에 나온 약 번호와 각각의 가장 높은 점수를 적는다.
- 출력
    - 확인 표: 제출 CSV 줄 수, 다시 예측한 대회 클래스 답 수, `others` 답 수(0.01 이상, 0.5 이상)
    - 사진당 자리 수: 자리가 0~2개, 3개, 4개, 5개 이상인 test 사진 수. 사진에는 알약이 3~4개이므로(9번 설명), 2개 이하는 못 찾은 알약이, 5개 이상은 겹친 박스나 배경 오탐이 있다는 신호다. 신호는 9-2에서 자세히 센다.
- 실행: 0번, 1번, 8-1을 실행한 뒤 실행한다. 8-1의 test 사진 목록(`TEST_IDS`), 제출 CSV 위치(`SUBMIT_DIR`), 점수 하한(`SUBMIT_CONF`)을 쓴다.

In [ ]:
from pill_detection.datasets import OTHERS_CATEGORY_ID  # others의 category_id
from pill_detection.evaluate import box_iou              # 두 xywh 박스의 IoU

CHECK_RUNS = ["v4_rtdetr-l_e10", "v4_rtdetr-l_e5"]  # 기준, 비교 (9번 설명)
CONF_HI = 0.5          # 자신 있게 낸 답 (1차 5-2와 같은 기준)
SAME_PILL_IOU = 0.5    # 같은 사진에서 이만큼 겹치면 같은 알약 (1차 5-4와 같은 기준)
PRED_COLS = ["image_id", "category_id", "bbox_x", "bbox_y", "bbox_w", "bbox_h", "score"]


def predict_others(run):
    """run의 best.pt로 test 사진을 다시 예측해 others 답만 표로 돌려준다. 대회 클래스 답의 수도 함께 돌려준다 (제출 CSV와 맞대 보기용)."""
    model = RTDETR(str(RUNS_EXP2 / run / "weights" / "best.pt"))
    results = model.predict(source=str(TEST_DIR), imgsz=COMMON["imgsz"], conf=SUBMIT_CONF, device=DEVICE,
                            stream=True, verbose=False)
    predictions = predictions_from_ultralytics(results)
    others = [[int(Path(p["file_name"]).stem), p["category_id"], *p["bbox"], p["score"]]
              for p in predictions if p["category_id"] == OTHERS_CATEGORY_ID]
    return pd.DataFrame(others, columns=PRED_COLS), len(predictions) - len(others)


def group_slots(pred):
    """사진마다 점수가 높은 박스부터 보고, 이미 있는 자리의 대표 박스와 IoU 0.5 이상이면 그 자리에, 아니면 새 자리에 넣는다."""
    rows = []
    for image_id, g in pred.sort_values("score", ascending=False).groupby("image_id"):
        slots = []
        for b in g.itertuples(index=False):
            box = [b.bbox_x, b.bbox_y, b.bbox_w, b.bbox_h]
            for s in slots:
                if box_iou(s["bbox"], box) >= SAME_PILL_IOU:
                    s["classes"].setdefault(int(b.category_id), float(b.score))  # 점수 순으로 보므로 처음 값이 그 약 번호의 가장 높은 점수
                    s["n_boxes"] += 1
                    break
            else:
                slots.append({"image_id": int(image_id), "category_id": int(b.category_id), "score": float(b.score), "bbox": box,
                              "classes": {int(b.category_id): float(b.score)}, "n_boxes": 1})
        rows.extend(slots)
    slots = pd.DataFrame(rows)
    slots["n_classes"] = slots["classes"].map(len)
    return slots


check_rows, slot_tables, others_tables = [], {}, {}
for run in CHECK_RUNS:
    start = time.perf_counter()
    sub = pd.read_csv(SUBMIT_DIR / f"exp2_{run}.csv")  # Kaggle에 올린 제출 CSV (8-1)
    others, n_competition = predict_others(run)
    torch.cuda.empty_cache()
    others_tables[run] = others
    check_rows.append({"run": run, "제출 CSV 줄": len(sub), "다시 예측: 대회 클래스 답": n_competition,
                       "다시 예측: others 답 (0.01 이상)": len(others), "others 답 (0.5 이상)": int((others["score"] >= CONF_HI).sum()),
                       "다시 예측 시간 (분)": round((time.perf_counter() - start) / 60, 1)})
    answers = pd.concat([sub[PRED_COLS], others], ignore_index=True)
    slot_tables[run] = group_slots(answers[answers["score"] >= CONF_HI])

print("확인: 제출 CSV와 다시 예측한 답 (others 답은 제출 CSV에 없다)")
display(pd.DataFrame(check_rows).set_index("run"))

count_rows = []
for run, slots in slot_tables.items():
    per_image = slots.groupby("image_id").size().reindex(sorted(TEST_IDS), fill_value=0)  # 자리가 없는 사진은 0
    count_rows.append({"run": run, "자리 0~2개": int((per_image <= 2).sum()), "3개": int((per_image == 3).sum()),
                       "4개": int((per_image == 4).sum()), "5개 이상": int((per_image >= 5).sum()), "자리 합계": len(slots)})
print(f"test 사진 {len(TEST_IDS)}장의 사진당 자리 수 (점수 {CONF_HI} 이상 답을 IoU {SAME_PILL_IOU} 이상끼리 묶음)")
display(pd.DataFrame(count_rows).set_index("run"))

### 9-2. 틀렸을 가능성이 큰 신호

- 9-1의 자리(점수 0.5 이상 답을 묶은 것)로, 정답 없이도 알 수 있는 모순을 센다. 기준 run은 `v4_rtdetr-l_e10`, 비교 run은 `v4_rtdetr-l_e5`다.
- 약 이름은 `classes.csv`의 이름이고, test 전용 18종에는 `[test 전용]`을 붙인다.

① 자리의 모순 (첫 번째 코드 셀)

| 신호 | 왜 틀렸을 가능성이 큰가 |
|---|---|
| 대표가 others인 자리 | test에는 대회 밖 알약이 없다(9-1). 자신 있게 낸 others는 틀린 답이다 |
| 약 번호가 둘 이상인 자리 | 한 알약을 두 약으로 동시에 자신 있게(0.5 이상) 봤다. 적어도 하나는 틀렸다 |
| 같은 약이 두 자리 이상인 사진 | 한 사진에 든 약은 모두 다르다(9번 설명). 둘 중 하나는 틀렸다 |
| 자리 5개 이상인 사진 | 알약은 3~4개다. 남는 자리는 겹친 박스나 배경 오탐이다. 그중 others 자리, 같은 약 두 자리가 있는 사진도 센다 |
| 두 run이 다르게 답한 자리 | 같은 사진에서 두 run의 자리를 대표 박스 IoU가 큰 쌍부터 짝짓고(0.5 이상), 대표 약 번호가 다른 쌍을 센다. 짝이 없는 자리도 센다 |

- 약 쌍 표: 많은 순으로 10개씩 보여 준다.
    - 약 번호가 둘 이상인 자리: (대표 약, 그다음으로 점수가 높은 약)
    - 두 run이 다르게 답한 자리: (기준 run의 약, 비교 run의 약)
- 같은 약이 두 자리 이상인 약: 어떤 약에서 그런 일이 생겼는지 많은 순으로 보여 준다.

② 약별 답한 자리 수와 test 알약 수 (두 번째 코드 셀)

- 팀 `classes.csv`의 `n_test`는 약마다 test에 나오는 알약 수다. 약마다 "모델이 그 약으로 답한 자리 수"와 나란히 놓으면, 어떤 약을 너무 많이 또는 너무 적게 답했는지 보인다.
    - 적게 답한 약: 그 약을 다른 약으로 봤을 가능성이 크다.
    - 많이 답한 약: 다른 약을 그 약으로 봤을 가능성이 크다.
- test 정보를 쓰는 분석이라, 원인을 이해하는 데만 쓰고 학습 데이터를 고르는 데는 쓰지 않는다.
- 그룹별 표: 56종, 18종, others마다 test 알약 수, 답한 자리 수, 약별 |차이|의 합
- 적어도 틀린 답의 수(|차이| 합 ÷ 2)
    - 알약 하나를 다른 약으로 답하면 두 약의 차이가 1씩 바뀌어 |차이|의 합이 많아야 2 늘어난다. 남는 자리나 못 찾은 알약은 1 늘린다.
    - 그래서 |차이| 합 ÷ 2는 "적어도 이만큼은 틀린 답이 있다"는 아래 한계다. 두 약을 서로 바꿔 답하면 차이가 0이 되므로, 실제로는 이보다 많을 수 있다.
- 약별 표: 기준 run의 |차이|가 큰 순으로 15종
- 한계: `n_test`는 팀 파일의 값이고, Kaggle public이 test 전체로 채점하는지는 모른다. 점수 0.5 미만으로만 답한 알약은 자리에 없다.
- 실행: 9-1을 실행한 뒤 ①, ② 순서로 실행한다. 파일만 읽는다.

In [ ]:
# ① 자리의 모순
CLASS_INFO = pd.read_csv(Path("data/datasets/classes.csv"), encoding="utf-8-sig").set_index("category_id")  # 읽기만 한다
BASE_RUN, COMPARE_RUN = CHECK_RUNS  # 기준 v4_e10, 비교 v4_e5 (9-1)
TOP_PAIRS = 10


def label(category_id):
    """표에 쓸 약 이름. 예: 3351 일양하이트린정 2mg, test 전용 18종은 [test 전용]을 붙인다."""
    if category_id == OTHERS_CATEGORY_ID:
        return "others"
    info = CLASS_INFO.loc[category_id]
    return f"{category_id} {info['name'][:14]}" + (" [test 전용]" if info["group"] == "test_only" else "")


def second_class(classes, top):
    """자리에 나온 약 번호 중 대표 다음으로 점수가 높은 약 번호"""
    rest = {c: s for c, s in classes.items() if c != top}
    return max(rest, key=rest.get) if rest else None


def match_slots(a, b):
    """같은 사진에서 두 run의 자리를 대표 박스 IoU가 큰 쌍부터 짝짓는다(IoU 0.5 이상). (짝, a에만 있는 자리, b에만 있는 자리)를 돌려준다."""
    groups_b = dict(tuple(b.groupby("image_id")))
    matched, used_b = [], set()
    for image_id, sa in a.groupby("image_id"):
        sb = groups_b.get(image_id, b.iloc[0:0])
        candidates = sorted(((box_iou(x["bbox"], y["bbox"]), i, j) for i, x in sa.iterrows() for j, y in sb.iterrows()), reverse=True)
        used_a = set()
        for iou, i, j in candidates:
            if iou < SAME_PILL_IOU:
                break
            if i in used_a or j in used_b:
                continue
            used_a.add(i)
            used_b.add(j)
            matched.append((i, j))
    only_a = sorted(set(a.index) - {i for i, _ in matched})
    only_b = sorted(set(b.index) - used_b)
    return matched, only_a, only_b


summary_rows, multi_pairs, dup_classes = [], {}, {}
for run, slots in slot_tables.items():
    per_image = slots.groupby("image_id").size()
    five = set(per_image[per_image >= 5].index)
    others_slots = slots[slots["category_id"] == OTHERS_CATEGORY_ID]
    multi = slots[slots["n_classes"] >= 2]
    dup = slots.groupby(["image_id", "category_id"]).size()
    dup = dup[dup >= 2].reset_index()
    summary_rows.append({"run": run, "자리 합계": len(slots), "대표가 others인 자리": len(others_slots),
                         "약 번호가 둘 이상인 자리": len(multi), "같은 약이 두 자리 이상인 사진": dup["image_id"].nunique(),
                         "자리 5개 이상인 사진": len(five),
                         "그중 others 자리가 있는 사진": len(five & set(others_slots["image_id"])),
                         "그중 같은 약 두 자리가 있는 사진": len(five & set(dup["image_id"]))})
    multi_pairs[run] = pd.Series([(r.category_id, second_class(r.classes, r.category_id)) for r in multi.itertuples()]).value_counts()
    dup_classes[run] = dup["category_id"].value_counts()
print("run별 신호 (점수 0.5 이상 답을 묶은 자리 기준)")
display(pd.DataFrame(summary_rows).set_index("run"))

matched, only_base, only_compare = match_slots(slot_tables[BASE_RUN], slot_tables[COMPARE_RUN])
pairs = pd.DataFrame([(slot_tables[BASE_RUN].loc[i, "category_id"], slot_tables[COMPARE_RUN].loc[j, "category_id"]) for i, j in matched],
                     columns=["기준", "비교"])
differ = pairs[pairs["기준"] != pairs["비교"]]
print(f"두 run 비교 ({BASE_RUN} 기준, {COMPARE_RUN} 비교)")
display(pd.DataFrame([{"짝지은 자리": len(pairs), "다르게 답한 자리": len(differ),
                       "기준에만 있는 자리": len(only_base), "비교에만 있는 자리": len(only_compare)}]))


def pair_table(counts, first, second):
    rows = [{first: label(a), second: label(b) if b is not None else "", "자리 수": n} for (a, b), n in counts.head(TOP_PAIRS).items()]
    return pd.DataFrame(rows)


print(f"약 번호가 둘 이상인 자리: 자주 나온 약 쌍 ({BASE_RUN})")
display(pair_table(multi_pairs[BASE_RUN], "대표 약", "그다음 약"))
print(f"두 run이 다르게 답한 자리: 자주 나온 약 쌍")
display(pair_table(differ.value_counts(), f"{BASE_RUN}의 답", f"{COMPARE_RUN}의 답"))
print("같은 약이 한 사진에 두 자리 이상 나온 약 (사진 수)")
display(pd.DataFrame({run: counts.rename(index=label) for run, counts in dup_classes.items()}).fillna(0).astype(int)
        .sort_values(BASE_RUN, ascending=False))

In [ ]:
# ② 약별 답한 자리 수와 test 알약 수 (원인을 이해하는 데만 쓴다. 학습 데이터를 고르는 데 쓰지 않는다)
GROUP_TEXT = {"kaggle": "Kaggle train 56종", "test_only": "test 전용 18종"}  # 7-3과 같다
SHORT = {BASE_RUN: "e10", COMPARE_RUN: "e5"}
TOP_CLASSES = 15

comp = CLASS_INFO[CLASS_INFO["competition"] == 1]
count_table = pd.DataFrame({"약": [label(c) for c in comp.index], "그룹": comp["group"].map(GROUP_TEXT).values,
                            "test 알약 수": comp["n_test"].astype(int).values}, index=comp.index)
for run, short in SHORT.items():
    answered = slot_tables[run]["category_id"].value_counts()
    count_table[f"{short} 자리"] = answered.reindex(comp.index, fill_value=0).astype(int)
    count_table[f"{short} 차이"] = count_table[f"{short} 자리"] - count_table["test 알약 수"]

group_rows = []
for run, short in SHORT.items():
    n_others = int((slot_tables[run]["category_id"] == OTHERS_CATEGORY_ID).sum())
    for group, text in GROUP_TEXT.items():
        part = count_table[count_table["그룹"] == text]
        group_rows.append({"run": run, "그룹": text, "test 알약 수": int(part["test 알약 수"].sum()),
                           "답한 자리 수": int(part[f"{short} 자리"].sum()), "약별 |차이| 합": int(part[f"{short} 차이"].abs().sum())})
    group_rows.append({"run": run, "그룹": "others", "test 알약 수": 0, "답한 자리 수": n_others, "약별 |차이| 합": n_others})
    total_diff = int(count_table[f"{short} 차이"].abs().sum()) + n_others
    group_rows.append({"run": run, "그룹": "합계", "test 알약 수": int(count_table["test 알약 수"].sum()),
                       "답한 자리 수": len(slot_tables[run]), "약별 |차이| 합": total_diff,
                       "적어도 틀린 답 (|차이| 합 ÷ 2)": total_diff / 2})
print("그룹별 답한 자리 수와 test 알약 수")
display(pd.DataFrame(group_rows).set_index(["run", "그룹"]))

top = count_table.reindex(count_table[f"{SHORT[BASE_RUN]} 차이"].abs().sort_values(ascending=False).index).head(TOP_CLASSES)
print(f"약별 답한 자리 수와 test 알약 수 ({BASE_RUN}의 |차이|가 큰 {TOP_CLASSES}종, 차이 = 자리 − test 알약 수)")
display(top.set_index("약"))

#### ③ 알약별로 맞았는지 (팀원의 test 검수표와 짝짓기)

- 왜 하는가: ②는 약별 수가 test와 거의 같다는 것까지만 보여 준다. 두 약을 서로 바꿔 답하면 수는 그대로라서, 알약 하나하나가 맞았는지는 알 수 없다.
- 검수표: `notebooks/eda/02_raewoo_test_crops.csv` (팀 EDA 노트북 02)
    - test 알약 3,229개가 한 줄씩 있다. 사진 번호, 위치(사분면), 박스, 사람이 눈으로 정한 약 번호(`final_category_id`)가 있다.
    - 약 번호는 팀원이 3,229개를 모두 눈으로 검수한 값이다(02 3-1, 3-2). Kaggle 정답 그 자체는 아니다.
    - 박스는 1차 RT-DETR(seed 1)의 예측 박스이고, 1차가 놓친 알약 24개는 눈으로 그린 대략의 박스다(02 1-2). 정답 박스가 아니라서, 알약의 위치를 짝짓는 데만 쓰고 박스가 정확한지는 판단하지 않는다.
- 짝짓기
    - 사진마다 검수표 알약과 9-1의 자리(점수 0.5 이상 답을 묶은 것)를 IoU가 큰 쌍부터 짝짓는다.
    - 기준은 IoU 0.3 이상이다. 눈으로 그린 박스는 대략의 값이라 9-1의 0.5보다 낮췄다. 한 사진의 알약은 사분면마다 떨어져 있어(02 1-1), 다른 알약과 잘못 짝지어질 가능성은 낮다.
- 알약마다 결과

| 결과 | 뜻 |
|---|---|
| 맞음 | 짝지은 자리의 대표 약 번호가 검수표의 약 번호와 같다 |
| 다른 약 | 짝지은 자리는 있지만 약 번호가 다르다(others 포함) |
| 못 찾음 | 짝지은 자리가 없다. 점수 0.5 이상으로는 답하지 않았다는 뜻이다 |

- 못 찾은 알약은 제출 CSV에서 그 알약 위치(IoU 0.3 이상)에 낸 정답 약의 가장 높은 점수를 함께 적는다. 0.5 아래의 낮은 점수로는 찾았는지 보기 위해서다.
- 남는 자리: 어느 알약과도 짝지어지지 않은 자리다. 배경이나, 같은 알약에 겹친 박스다.
- 출력
    - run별, 그룹별 결과 수
    - 다른 약으로 답한 쌍(검수표 약 → 답한 약)
    - 못 찾은 알약과 그 알약의 정답 약 점수
    - 남는 자리의 약
- 원인을 이해하는 데만 쓰고, 학습 데이터를 고르는 데는 쓰지 않는다(②와 같은 조건).
- 알약별 결과(`pill_check`)와 남는 자리(`extra_slots`)는 9-3에서 사진을 고를 때 쓴다.
- 실행: ②를 실행한 뒤 실행한다. 파일만 읽는다.

In [ ]:
# ③ 알약별로 맞았는지: 팀원의 test 검수표와 짝짓기 (원인을 이해하는 데만 쓴다. 학습 데이터를 고르는 데 쓰지 않는다)
TEST_CROPS_CSV = Path("notebooks/eda/02_raewoo_test_crops.csv")  # 팀원이 test 알약 3,229개를 눈으로 검수한 표 (읽기만 한다)
MATCH_IOU = 0.3   # 검수표 박스는 대략의 값이 섞여 있어 9-1의 0.5보다 낮춘다

test_crops = pd.read_csv(TEST_CROPS_CSV, encoding="utf-8-sig")
test_crops["image_id"] = test_crops["test_file"].str.replace(".png", "", regex=False).astype(int)
test_crops["bbox"] = test_crops["bbox"].map(json.loads)  # "[x, y, w, h]" 글자 → 리스트
assert test_crops["final_category_id"].notna().all(), "검수표에 약 번호가 빈 알약이 있습니다."
assert len(test_crops) == int(comp["n_test"].sum()), "검수표의 알약 수가 classes.csv의 n_test 합과 다릅니다."


def match_to_crops(slots):
    """사진마다 검수표 알약과 자리를 IoU가 큰 쌍부터 짝짓는다(IoU 0.3 이상). {검수표 줄: (자리, IoU)}를 돌려준다."""
    slot_groups = dict(tuple(slots.groupby("image_id")))
    matched = {}
    for image_id, crop_group in test_crops.groupby("image_id"):
        slot_group = slot_groups.get(image_id, slots.iloc[0:0])
        candidates = sorted(((box_iou(c["bbox"], s["bbox"]), ci, si)
                             for ci, c in crop_group.iterrows() for si, s in slot_group.iterrows()), reverse=True)
        used_slots = set()
        for iou, ci, si in candidates:
            if iou < MATCH_IOU:
                break
            if ci in matched or si in used_slots:
                continue
            matched[ci] = (si, iou)
            used_slots.add(si)
    return matched


def best_truth_score(sub_groups, image_id, truth, box):
    """제출 CSV에서 그 알약 위치(IoU 0.3 이상)에 낸 정답 약의 가장 높은 점수. 없으면 0"""
    group = sub_groups.get(image_id)
    if group is None:
        return 0.0
    group = group[group["category_id"] == truth]
    return max((r.score for r in group.itertuples() if box_iou([r.bbox_x, r.bbox_y, r.bbox_w, r.bbox_h], box) >= MATCH_IOU),
               default=0.0)


pill_check, extra_slots = {}, {}
for run, slots in slot_tables.items():
    matched = match_to_crops(slots)
    sub_groups = dict(tuple(pd.read_csv(SUBMIT_DIR / f"exp2_{run}.csv").groupby("image_id")))  # 8-1의 제출 CSV
    rows = []
    for ci, c in test_crops.iterrows():
        truth = int(c["final_category_id"])
        si, iou = matched.get(ci, (None, None))
        answer = None if si is None else int(slots.at[si, "category_id"])
        rows.append({"crop_id": c["crop_id"], "image_id": c["image_id"], "위치": c["position"], "검수표 약": truth,
                     "그룹": GROUP_TEXT[CLASS_INFO.at[truth, "group"]], "답한 약": answer,
                     "점수": None if si is None else slots.at[si, "score"], "IoU": iou, "자리": si,
                     "결과": "못 찾음" if si is None else ("맞음" if answer == truth else "다른 약"),
                     "정답 약 최고 점수 (제출 CSV)": best_truth_score(sub_groups, c["image_id"], truth, c["bbox"]) if si is None else None})
    pill_check[run] = pd.DataFrame(rows)
    extra_slots[run] = slots.drop(index=[si for si, _ in matched.values()])

# run별, 그룹별 결과 수
result_rows = []
for run, check in pill_check.items():
    for group in [*GROUP_TEXT.values(), "합계"]:
        part = check if group == "합계" else check[check["그룹"] == group]
        counts = part["결과"].value_counts()
        result_rows.append({"run": run, "그룹": group, "test 알약": len(part), "맞음": int(counts.get("맞음", 0)),
                            "다른 약": int(counts.get("다른 약", 0)), "못 찾음": int(counts.get("못 찾음", 0)),
                            "맞은 비율": counts.get("맞음", 0) / len(part),
                            "남는 자리": len(extra_slots[run]) if group == "합계" else None})
print(f"알약별 결과 (검수표 {len(test_crops)}개, 짝짓기 IoU {MATCH_IOU} 이상)")
display(pd.DataFrame(result_rows).set_index(["run", "그룹"]).round(4))

# 다른 약으로 답한 쌍
wrong = pd.concat([check[check["결과"] == "다른 약"].assign(run=run) for run, check in pill_check.items()])
if len(wrong):
    wrong_pairs = wrong.groupby(["검수표 약", "답한 약", "run"]).size().unstack("run", fill_value=0)
    wrong_pairs.index = [f"{label(a)} → {label(b)}" for a, b in wrong_pairs.index]
    print("다른 약으로 답한 알약 (검수표 약 → 답한 약, 알약 수)")
    display(wrong_pairs)
else:
    print("다른 약으로 답한 알약: 없음")

# 못 찾은 알약
missed = pd.concat([check[check["결과"] == "못 찾음"].assign(run=run) for run, check in pill_check.items()])
if len(missed):
    missed_view = missed.assign(약=missed["검수표 약"].map(label))[["run", "image_id", "위치", "약", "정답 약 최고 점수 (제출 CSV)"]]
    print("못 찾은 알약 (점수 0.5 이상 자리가 없는 알약)")
    display(missed_view.sort_values(["run", "약"]).reset_index(drop=True).round(4))
else:
    print("못 찾은 알약: 없음")

# 남는 자리의 약
extra_counts = pd.DataFrame({run: slots["category_id"].map(label).value_counts() for run, slots in extra_slots.items()}).fillna(0).astype(int)
print("남는 자리 (어느 검수표 알약과도 짝지어지지 않은 자리)의 대표 약")
display(extra_counts.sort_values(BASE_RUN, ascending=False) if len(extra_counts) else "없음")

#### ④ test 사진 크기와 좌표 기준

- 왜 하는가: ③에서 v4는 test 알약을 모두 찾고 99.8%를 맞는 약으로 답했다(팀원의 검수표 기준). 그런데도 Kaggle은 0.61이다. 팀의 다른 모델도 val은 0.9 이상, Kaggle은 0.6 안팎이다(팀원 확인). 모델과 상관없이 비슷하므로, 약을 맞힌 뒤의 박스나 정답, 채점 쪽을 의심한다.
- 그중 가장 싸게 확인할 수 있는 것이 좌표 기준이다.
    - 예측 박스는 우리가 받은 test PNG의 픽셀 좌표다(8-1).
    - test 정답 박스가 다른 크기의 사진에서 매겨졌다면, 모든 박스가 같은 비율로 조금씩 어긋난다. 그러면 약을 다 맞혀도 높은 IoU 기준(0.85~0.95)에서 대부분 틀린 것으로 채점된다.
- 확인하는 것
    - 사진 크기(가로 × 세로): test PNG 842장, 대회 train PNG 232장, 2차 val 사진 1,000장
    - 대회 train의 좌표 기준: 정제한 Kaggle 라벨 JSON(`data/processed/train_annotations`)의 `images[].width`, `height`가 실제 PNG 크기와 같은지. 같다면 대회 정답 박스는 PNG 픽셀 좌표로 매겨진 것이다.
    - 예측 박스가 사진 밖으로 나간 정도: 9-1의 자리(점수 0.5 이상) 중 박스가 사진 밖으로 나간 자리 수와, 가장 많이 나간 픽셀. 8-1은 박스를 사진 안으로 자르지 않았다.
- 읽는 법
    - test PNG가 train PNG, JSON 기준과 모두 같은 크기라면: 좌표 기준이 어긋났을 가능성은 낮다. 남는 후보는 test 정답 박스를 그린 방식, test 정답의 오류, Kaggle 채점 방식이다.
    - test PNG만 크기가 다르면: 좌표 기준이 어긋났을 가능성이 크다. 어떤 비율로 다른지 보고 다음 단계를 정한다.
- 실행: 9-1을 실행한 뒤 실행한다. 사진 파일의 머리 부분만 읽어서 몇 초면 끝난다.

In [ ]:
# ④ test 사진 크기와 좌표 기준
from collections import Counter

from PIL import Image  # Ultralytics와 함께 설치되어 있다. 사진 크기는 파일 머리만 읽는다

TRAIN_IMAGE_DIR = Path("data/raw/train_images")             # 대회 train PNG 232장
TRAIN_ANN_DIR = Path("data/processed/train_annotations")    # 정제한 Kaggle 라벨 JSON (EXPERIMENTS.md 2단계)
VAL_IMAGE_DIR = EXP2_DIR / "v4" / "images" / "val"          # 2차 val 1,000장 (세 방법이 같다)


def image_sizes(folder):
    """폴더의 PNG마다 (가로, 세로)"""
    sizes = {}
    for path in sorted(Path(folder).glob("*.png")):
        with Image.open(path) as img:
            sizes[path.name] = img.size
    return sizes


size_sets = {"test PNG": image_sizes(TEST_DIR), "대회 train PNG": image_sizes(TRAIN_IMAGE_DIR), "2차 val 사진": image_sizes(VAL_IMAGE_DIR)}
json_sizes = {}
for path in sorted(TRAIN_ANN_DIR.rglob("*.json")):
    for image in json.loads(path.read_text(encoding="utf-8-sig"))["images"]:
        json_sizes[image["file_name"]] = (image["width"], image["height"])
size_sets["대회 train 라벨 JSON의 width, height"] = json_sizes

size_rows = []
for name, sizes in size_sets.items():
    counts = Counter(sizes.values())
    size_rows.append({"집합": name, "수": len(sizes), "크기 종류": len(counts),
                      "크기 (가로 × 세로): 장 수": ", ".join(f"{w} × {h}: {n}" for (w, h), n in counts.most_common(5))})
print("사진 크기")
display(pd.DataFrame(size_rows).set_index("집합"))

train_png = size_sets["대회 train PNG"]
same = sum(1 for name, size in train_png.items() if json_sizes.get(name) == size)
print(f"대회 train PNG와 그 라벨 JSON의 width, height가 같은 사진: {same} / {len(train_png)}장"
      f" (JSON이 없는 PNG {sum(1 for name in train_png if name not in json_sizes)}장)")

# 예측 박스가 사진 밖으로 나간 정도 (9-1의 자리, 점수 0.5 이상)
test_sizes = {int(Path(name).stem): size for name, size in size_sets["test PNG"].items()}
out_rows = []
for run, slots in slot_tables.items():
    over = []
    for s in slots.itertuples():
        w_img, h_img = test_sizes[s.image_id]
        x, y, w, h = s.bbox
        over.append(max(0.0, -x, -y, x + w - w_img, y + h - h_img))
    over = pd.Series(over)
    out_rows.append({"run": run, "자리": len(slots), "사진 밖으로 나간 자리": int((over > 0).sum()),
                     "1픽셀 넘게 나간 자리": int((over > 1).sum()), "가장 많이 나간 픽셀": round(float(over.max()), 1)})
print("예측 박스가 사진 밖으로 나간 정도 (9-1의 자리)")
display(pd.DataFrame(out_rows).set_index("run"))

### 9-3. 진단 제출: Kaggle 채점이 남는 답을 감점하는가

- 왜 하는가
    - 9-2에서 v4는 test 알약을 모두 찾고 99.8%를 맞는 약으로 답했다(③). test 사진의 크기와 좌표 기준도 train과 같았다(④).
    - 팀의 다른 모델도 val은 0.9 이상, Kaggle은 0.6 안팎이다(팀원 확인). 모델이 달라도 같은 곳에서 막히므로, 모든 모델에 공통인 것을 의심한다.
    - 공통인 것 중 하나가 제출 방식이다. 팀 가이드대로 모두 점수 0.01 이상 답을 다 낸다. 사진당 200개 안팎인데, 알약은 3~4개다.
- 두 가지 가능성
    - 표준 COCO mAP라면: 클래스마다 답을 점수 순으로 줄 세워 AP를 매기므로, 맞는 답보다 점수가 낮은 남는 답은 점수를 거의 깎지 않는다(1차 노트북 5-1). 이 경우 0.5 이상만 내도 점수가 거의 같다.
    - Kaggle 채점 코드가 사진당 답의 수를 제한하거나 남는 답을 감점한다면: 0.5 이상만 내면 점수가 크게 오른다.
- 만드는 것: 8-1의 `exp2_v4_rtdetr-l_e5.csv`(Kaggle 0.61558)에서 점수 0.5 이상인 줄만 남긴 CSV
    - 다시 예측하지 않는다. `annotation_id`만 1부터 다시 매긴다.
    - 저장: `submissions/exp2_v4_rtdetr-l_e5_conf050.csv`
    - v4_e5를 쓰는 이유: 남는 자리가 2개뿐이라(9-2 ③), 0.5 이상만 남기면 거의 알약 하나에 한 줄이 된다. 5개 run 중 Kaggle 점수가 가장 높은 run이기도 하다.
- 해석 기준 (제출 전에 정한다)

| 결과 (0.61558과의 차이) | 해석 |
|---|---|
| 0.004 미만 | 채점은 표준 mAP와 비슷하게 동작한다. 남는 원인은 test 정답 박스 쪽이다 |
| 0.004 이상 오른다 | Kaggle 채점이 남는 답을 감점한다. 팀 전체의 제출 방식을 바꿔야 한다 |
| 0.004 이상 떨어진다 | 낮은 점수 답이 실제로 점수를 올리고 있었다. 지금 방식이 맞다 |

- 0.004는 1차 모델의 Kaggle 점수가 seed만 바꿔도 움직인 폭이다(8번 설명).
- run을 고르기 위한 제출이 아니라 채점 방식을 확인하기 위한 제출이다. 결과가 팀 전체의 제출 방식에 영향을 줄 수 있어서, 팀에 알리고 하루 제출 횟수 중 1회만 쓴다.
- 확인 출력: 8-1과 같은 형식 확인(열, `annotation_id`, `image_id`, `category_id`), 원본과 남긴 줄 수, 사진당 줄 수의 분포, 가장 낮은 점수
- 실행: 0번, 1번, 8-1을 실행한 뒤 실행한다. 제출 CSV만 읽고 새 CSV 하나를 쓴다.

In [ ]:
DIAG_RUN = "v4_rtdetr-l_e5"   # 남는 자리가 가장 적은 run (9-2 ③)
DIAG_CONF = 0.5               # 이 점수 이상인 답만 남긴다 (9-1의 "자신 있게 낸 답" 기준)
diag_src = SUBMIT_DIR / f"exp2_{DIAG_RUN}.csv"            # Kaggle에 올린 제출 CSV (0.61558)
diag_dst = SUBMIT_DIR / f"exp2_{DIAG_RUN}_conf050.csv"    # 진단 제출용

original = pd.read_csv(diag_src)
diag = original[original["score"] >= DIAG_CONF].copy()   # 순서(사진 번호, 점수 높은 순)는 원본 그대로
diag["annotation_id"] = range(1, len(diag) + 1)
diag = diag[SUBMISSION_COLUMNS]
diag.to_csv(diag_dst, index=False, encoding="utf-8", lineterminator="\n")

# 형식 확인: 8-1과 같다. 하나라도 어긋나면 멈춘다
saved = pd.read_csv(diag_dst)
assert list(saved.columns) == SUBMISSION_COLUMNS, f"열이 제출 형식과 다릅니다: {list(saved.columns)}"
assert saved["annotation_id"].tolist() == list(range(1, len(saved) + 1)), "annotation_id가 1부터 차례대로가 아닙니다."
assert set(saved["image_id"]) <= TEST_IDS, "test 파일 번호가 아닌 image_id가 있습니다."
assert set(saved["category_id"]) <= competition_from_classes(EXP2_DIR / "v4" / "classes.csv"), "대회 클래스가 아닌 category_id가 있습니다."

per_image = saved.groupby("image_id").size().reindex(sorted(TEST_IDS), fill_value=0)
print("저장:", diag_dst.as_posix())
display(pd.DataFrame([{"원본 줄 수": len(original), "남긴 줄 수": len(saved),
                       "답이 있는 사진": f"{int((per_image > 0).sum())}/{len(TEST_IDS)}",
                       "사진당 0줄": int((per_image == 0).sum()), "1~2줄": int(per_image.between(1, 2).sum()),
                       "3줄": int((per_image == 3).sum()), "4줄": int((per_image == 4).sum()), "5줄 이상": int((per_image >= 5).sum()),
                       "가장 낮은 점수": float(saved["score"].min())}]))

### 9번 중간 정리 (진단 제출 전)

- 상태: 2026-11-02의 하루 제출 횟수(팀 10회)를 모두 써서, 9-3의 진단 제출은 다음 날 한다. 9-3 셀은 진단용 CSV만 만든다.

진단용 CSV와 이미 낸 CSV의 차이

| | 이미 낸 `exp2_v4_rtdetr-l_e5.csv` | 진단용 `exp2_v4_rtdetr-l_e5_conf050.csv` |
|---|---|---|
| 모델, 박스, 약 번호, 점수 | v4_e5의 답 | 같다(같은 답에서 줄만 뺀다) |
| 남긴 답 | 점수 0.01 이상 전부(173,409줄, 사진당 200개 안팎) | 점수 0.5 이상만(사진당 3~4개) |
| `annotation_id` | 1부터 | 남긴 줄에 1부터 다시 매김 |

- 달라지는 것은 낮은 점수 답을 같이 내느냐뿐이다. 점수가 바뀐다면 원인은 Kaggle이 낮은 점수 답을 채점하는 방식밖에 없다.

val과 Kaggle의 차이: 지금까지 확인한 것 (9-2)

- test 알약 3,229개를 모두 찾고, 99.8%를 맞는 약으로 답했다(③, 팀원의 test 검수표 기준). test 전용 18종은 100%다.
- test 사진은 train과 같은 크기(976 × 1280)이고, 예측 박스가 사진 밖으로 나간 것은 많아야 1.3픽셀이다(④).
- 팀의 다른 모델도 val은 0.9 이상, Kaggle은 0.6 안팎이다(팀원 확인).
- 그래서 약을 못 찾거나 잘못 알아본 것은 원인이 아니다. 9번 설명의 후보 1(대회 사진 속 18종), 2(비슷한 약끼리 헷갈림)는 해당하지 않고, 모든 모델에 공통인 원인이 남는다.

남은 후보 (모두 확인 전)

| 후보 | 내용 | 확인 방법 |
|---|---|---|
| ① Kaggle 채점이 낮은 점수 답을 감점 | 표준 COCO mAP라면 낮은 점수 답은 거의 영향이 없다. Kaggle 채점 코드가 사진당 답 수를 제한하거나 남는 답을 감점한다면, 사진당 200개 안팎을 내는 팀 전체가 똑같이 손해를 본다 | 9-3 진단 제출 |
| ② test 정답 박스가 다르게 그려졌거나 오류가 있음 | 약을 맞혀도 박스가 test 정답과 조금씩 어긋나면 IoU 0.85~0.95에서 틀린 것으로 채점된다. val의 대회 사진 47장에서는 박스가 잘 맞았으므로(`kaggle` 0.98~0.997), 그렇다면 test 정답만 다르다는 뜻이다 | test 정답이 없어 직접 확인할 수 없다 |
| ③ Kaggle 채점의 세부 방식 | 어떤 IoU 기준과 클래스 평균 방식으로 매기는지, public이 test의 어느 범위인지 | 대회 페이지의 Evaluation 설명과 토론 |

- 9번 설명에서는 후보 4(정답 박스의 차이)를 반대 근거가 있어 우선순위를 낮췄다. 9-2에서 약 구분과 좌표 기준이 원인이 아님을 확인했으므로, 지금은 ②가 남은 유력 후보 중 하나다.

Kaggle 점수를 올리는 방향 (원인에 따라 다르다)

- ①이 맞다면(진단 제출이 0.61558보다 0.004 이상 오르면)
    - 모델은 그대로 두고, 제출할 때 자신 있는 답만 낸다.
    - 점수 기준(0.3, 0.5, 0.7 등)은 Kaggle이 아니라 val에서 같은 방식으로 채점해 정한다.
    - 팀 가이드의 제출 방식(점수 0.01 이상)을 바꿔야 하는 결과라 팀과 공유한다.
- ②가 맞다면(차이가 0.004 미만이면)
    - 약 구분은 이미 99.8%라서, 데이터, epoch, 방법(v2/v3/v4)을 바꿔도 크게 오르지 않는다. 8-2의 5개 제출의 차이도 0.013 안이었다.
    - 오를 여지는 박스 정밀도다. 다만 test 정답 박스가 어떻게 그려졌는지 모르면 맞추기 어렵다.
    - 박스를 조금 키우거나 줄여 내 보는 방법은 리더보드에 맞추는 성격이 강해서 팀과 상의해서 정한다.
    - 입력 크기를 640보다 키우거나 여러 모델의 박스를 합치는 방법도 있다. 다만 val에서 이미 박스가 잘 맞고 있어 test에서 얼마나 도움이 될지는 불확실하다.
- ③은 대회 설명을 확인한 뒤 판단한다.

진단 제출 전에 할 것

- 팀원에게 묻기: 점수 기준(conf)을 0.25나 0.5처럼 높여 제출한 적이 있는지, 그때 점수가 몇이었는지. 있다면 진단 제출 없이도 ①을 판단할 수 있다.
- 대회 페이지 확인: Evaluation 설명이나 토론에 사진당 최대 답 수나 채점 코드에 대한 언급이 있는지
- 9-3 실행: 진단용 CSV가 사진 842장 모두에 사진당 3~4줄로 만들어졌는지 확인